# Section 1.0 — Foundation Bootstrap & Run Identity

This section initializes the Data Foundation from the verified project setup.
It loads the existing path registry, setup summary, source fingerprints, and frozen fold manifest.
Critical registered paths and required setup flags are checked before any data analysis begins.
The frozen fold file is only checked for readability and required columns here.
Full fold integrity and leakage validation will be performed later in Section 1.9.
A unique run ID, timestamp, Git branch, commit, and working-tree status are recorded for reproducibility.
Python 3.12 and CUDA are not blocking checks at this stage.
No training data profiling, transcript scanning, or turn parsing is performed here.
The section ends with one hard gate: `FOUNDATION_BOOTSTRAP_READY`.


In [1]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import pandas as pd

PROJECT_ROOT_HINT = Path(r"C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition")

registry_candidates = [
    Path.cwd() / "scratch_mastery_outputs" / "00_project_setup" / "path_registry.json",
    *[p / "scratch_mastery_outputs" / "00_project_setup" / "path_registry.json" for p in Path.cwd().parents],
    PROJECT_ROOT_HINT / "scratch_mastery_outputs" / "00_project_setup" / "path_registry.json",
]

PATH_REGISTRY_PATH = next((p for p in registry_candidates if p.exists()), None)
assert PATH_REGISTRY_PATH is not None, "path_registry.json not found. Run 00_environment_and_paths.ipynb first."

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

path_registry = load_json(PATH_REGISTRY_PATH)

PROJECT_ROOT = Path(path_registry["project_root"])
SETUP_OUTPUT_DIR = Path(path_registry["setup_output_dir"])
SCRATCH_OUTPUT_ROOT = Path(path_registry["scratch_output_root"])

SETUP_SUMMARY_PATH = SETUP_OUTPUT_DIR / "setup_summary.json"
SOURCE_FINGERPRINTS_PATH = SETUP_OUTPUT_DIR / "source_fingerprints.json"
FROZEN_FOLD_PATH = Path(path_registry["frozen_fold_manifest_path"])

PHASE1_ROOT = SCRATCH_OUTPUT_ROOT / "01_data_foundation"
INVENTORY_OUTPUT_DIR = PHASE1_ROOT / "01_inventory"

PHASE1_ROOT.mkdir(parents=True, exist_ok=True)
INVENTORY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

setup_summary = load_json(SETUP_SUMMARY_PATH)
source_fingerprints = load_json(SOURCE_FINGERPRINTS_PATH)

In [2]:
def path_check(name, path, required=True):
    return {
        "check": name,
        "required": required,
        "passed": Path(path).exists(),
        "detail": str(path),
    }

def flag_check(name, value, required=True):
    return {
        "check": name,
        "required": required,
        "passed": bool(value),
        "detail": bool(value),
    }

critical_paths = {
    "Project root": PROJECT_ROOT,
    "Data root": Path(path_registry["data_root"]),
    "Advanced notebook root": Path(path_registry["advanced_notebook_root"]),
    "Scratch output root": SCRATCH_OUTPUT_ROOT,
    "Train features": Path(path_registry["train_features_path"]),
    "Train labels": Path(path_registry["train_labels_path"]),
    "Frozen fold manifest": FROZEN_FOLD_PATH,
}

if path_registry["transcript_source_type"] == "external_path":
    critical_paths["Transcript source"] = Path(path_registry["transcript_source"])

checks = [path_check(name, path) for name, path in critical_paths.items()]

required_flags = [
    "data_paths_ready",
    "feature_label_schema_ready",
    "frozen_folds_ready",
    "transcript_source_ready",
    "scratch_output_git_safe",
]

checks += [flag_check(flag, setup_summary.get(flag)) for flag in required_flags]

checks += [
    flag_check("python_312_ready", setup_summary.get("python_312_ready"), required=False),
    flag_check("cuda_ready", setup_summary.get("cuda_ready"), required=False),
]

fingerprint_ok = all(
    source_fingerprints.get(key)
    for key in ["train_features_sha256", "train_labels_sha256"]
)

checks.append(flag_check("source_fingerprints_ready", fingerprint_ok))

frozen_folds = pd.read_parquet(FROZEN_FOLD_PATH)
fold_columns_ok = {"response_id", "session_id", "fold"}.issubset(frozen_folds.columns)

checks.append(flag_check("frozen_fold_structure", len(frozen_folds) > 0 and fold_columns_ok))

bootstrap_checks = pd.DataFrame(checks)
display(bootstrap_checks)

,check,required,passed,detail
0,Project root,True,True,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...
1,Data root,True,True,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...
2,Advanced notebook root,True,True,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...
3,Scratch output root,True,True,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...
4,Train features,True,True,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...
5,Train labels,True,True,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...
6,Frozen fold manifest,True,True,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...
7,Transcript source,True,True,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...
8,data_paths_ready,True,True,True
9,feature_label_schema_ready,True,True,True


In [3]:
def git_value(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip() if result.returncode == 0 else None

run_timestamp = datetime.now().astimezone()
RUN_ID = f"DF_{run_timestamp.strftime('%Y%m%d_%H%M%S')}"

git_branch = git_value("branch", "--show-current")
git_commit = git_value("rev-parse", "HEAD")
git_status = git_value("status", "--short")

FOUNDATION_CONTEXT = {
    "phase": "data_foundation",
    "contract_version": "1.0",
    "run_id": RUN_ID,
    "run_timestamp": run_timestamp.isoformat(),
    "git_branch": git_branch,
    "git_commit": git_commit,
    "git_dirty": bool(git_status),
}

required_failures = bootstrap_checks[
    bootstrap_checks["required"] & ~bootstrap_checks["passed"]
]

FOUNDATION_BOOTSTRAP_READY = required_failures.empty

summary = pd.DataFrame({
    "item": [
        "Run ID",
        "Frozen fold rows",
        "Required checks",
        "Required failures",
        "Git dirty",
        "FOUNDATION_BOOTSTRAP_READY",
    ],
    "value": [
        RUN_ID,
        len(frozen_folds),
        int(bootstrap_checks["required"].sum()),
        len(required_failures),
        bool(git_status),
        FOUNDATION_BOOTSTRAP_READY,
    ],
})

display(summary)

assert FOUNDATION_BOOTSTRAP_READY, (
    "Section 1.0 failed.\n\n"
    + required_failures[["check", "detail"]].to_string(index=False)
)

,item,value
0,Run ID,DF_20260808_234758
1,Frozen fold rows,35072
2,Required checks,15
3,Required failures,0
4,Git dirty,True
5,FOUNDATION_BOOTSTRAP_READY,True


# Section 1.1 — Authoritative Source Mapping

This section defines which project sources are allowed to build the new Data Foundation.
Official training features, labels, raw transcripts, and frozen folds are treated as authoritative sources.
Submission-format files are kept only as inference-schema references.
Previous master datasets, baseline predictions, and old parser outputs are reference-only artifacts.
Reference-only artifacts must never be used to reconstruct the new canonical foundation.
Each source is assigned a clear authority role and permitted foundation use.
Required authoritative sources must exist before the inventory can continue.
The section ends with the `SOURCE_AUTHORITY_READY` gate.

In [4]:
def make_source(source_id, source_name, source_class, authority_for, path=None, required_now=False, foundation_use="VALIDATE_ONLY", note=""):
    path = Path(path) if path else None

    return {
        "source_id": source_id,
        "source_name": source_name,
        "source_class": source_class,
        "authority_for": authority_for,
        "foundation_use": foundation_use,
        "required_now": required_now,
        "path": str(path) if path else None,
        "exists": path.exists() if path else None,
        "note": note,
    }

sources = [
    make_source(
        "SRC_TRAIN_FEATURES",
        "Train features",
        "AUTHORITATIVE",
        "response_session_objective_metadata",
        path_registry["train_features_path"],
        True,
        "BUILD"
    ),
    make_source(
        "SRC_TRAIN_LABELS",
        "Train labels",
        "AUTHORITATIVE",
        "target_labels",
        path_registry["train_labels_path"],
        True,
        "BUILD"
    ),
    make_source(
        "SRC_FROZEN_FOLDS",
        "Frozen fold manifest",
        "AUTHORITATIVE",
        "validation_fold_assignment",
        path_registry["frozen_fold_manifest_path"],
        True,
        "ATTACH"
    ),
]

transcript_type = path_registry.get("transcript_source_type")

if transcript_type == "external_path":
    sources.append(
        make_source(
            "SRC_TRANSCRIPTS",
            "Raw transcripts",
            "AUTHORITATIVE",
            "raw_dialogue",
            path_registry["transcript_source"],
            True,
            "BUILD",
            "External transcript directory"
        )
    )
else:
    sources.append(
        make_source(
            "SRC_TRANSCRIPTS",
            "Raw transcripts",
            "AUTHORITATIVE",
            "raw_dialogue",
            path_registry["train_features_path"],
            True,
            "BUILD",
            f"Embedded transcript column: {path_registry.get('embedded_transcript_column')}"
        )
    )

submission_keys = sorted(
    key for key in path_registry
    if key.startswith("submission_format_path")
)

for i, key in enumerate(submission_keys, start=1):
    sources.append(
        make_source(
            f"SRC_SUBMISSION_{i}",
            f"Submission format {i}",
            "SCHEMA_REFERENCE",
            "future_inference_schema",
            path_registry[key],
            False,
            "SCHEMA_ONLY"
        )
    )

source_registry = pd.DataFrame(sources)

In [5]:
reference_sources = [
    make_source(
        "REF_OLD_MASTER",
        "Previous master dataset",
        "REFERENCE_ONLY",
        "historical_derived_dataset",
        foundation_use="DO_NOT_BUILD_FROM",
        note="May be used only for comparison or historical reconciliation"
    ),
    make_source(
        "REF_BASELINE_OOF",
        "Baseline OOF predictions",
        "REFERENCE_ONLY",
        "historical_model_diagnostics",
        foundation_use="DO_NOT_BUILD_FROM",
        note="Diagnostic reference only"
    ),
    make_source(
        "REF_OLD_PARSER",
        "Previous parser outputs",
        "REFERENCE_ONLY",
        "historical_parser_outputs",
        foundation_use="DO_NOT_BUILD_FROM",
        note="Must not define the new canonical turn structure"
    ),
]

source_registry = pd.concat(
    [source_registry, pd.DataFrame(reference_sources)],
    ignore_index=True
)

display(
    source_registry[
        [
            "source_id",
            "source_name",
            "source_class",
            "authority_for",
            "foundation_use",
            "required_now",
            "exists",
        ]
    ]
)

,source_id,source_name,source_class,authority_for,foundation_use,required_now,exists
0,SRC_TRAIN_FEATURES,Train features,AUTHORITATIVE,response_session_objective_metadata,BUILD,True,True
1,SRC_TRAIN_LABELS,Train labels,AUTHORITATIVE,target_labels,BUILD,True,True
2,SRC_FROZEN_FOLDS,Frozen fold manifest,AUTHORITATIVE,validation_fold_assignment,ATTACH,True,True
3,SRC_TRANSCRIPTS,Raw transcripts,AUTHORITATIVE,raw_dialogue,BUILD,True,True
4,SRC_SUBMISSION_1,Submission format 1,SCHEMA_REFERENCE,future_inference_schema,SCHEMA_ONLY,False,True
5,SRC_SUBMISSION_2,Submission format 2,SCHEMA_REFERENCE,future_inference_schema,SCHEMA_ONLY,False,True
6,REF_OLD_MASTER,Previous master dataset,REFERENCE_ONLY,historical_derived_dataset,DO_NOT_BUILD_FROM,False,None
7,REF_BASELINE_OOF,Baseline OOF predictions,REFERENCE_ONLY,historical_model_diagnostics,DO_NOT_BUILD_FROM,False,None
8,REF_OLD_PARSER,Previous parser outputs,REFERENCE_ONLY,historical_parser_outputs,DO_NOT_BUILD_FROM,False,None


In [6]:
required_authorities = {
    "response_session_objective_metadata",
    "target_labels",
    "raw_dialogue",
    "validation_fold_assignment",
}

required_sources = source_registry[source_registry["required_now"]]
missing_required = required_sources[required_sources["exists"] != True]

available_authorities = set(
    source_registry.loc[
        source_registry["source_class"] == "AUTHORITATIVE",
        "authority_for"
    ]
)

missing_authorities = required_authorities - available_authorities

invalid_reference_use = source_registry[
    (source_registry["source_class"] == "REFERENCE_ONLY") &
    (source_registry["foundation_use"] != "DO_NOT_BUILD_FROM")
]

duplicate_source_ids = source_registry["source_id"].duplicated().sum()

SOURCE_AUTHORITY_READY = (
    missing_required.empty
    and not missing_authorities
    and invalid_reference_use.empty
    and duplicate_source_ids == 0
)

authority_summary = pd.DataFrame({
    "item": [
        "Registered sources",
        "Authoritative sources",
        "Schema references",
        "Reference-only sources",
        "Missing required sources",
        "Missing authority roles",
        "Invalid reference usage",
        "Duplicate source IDs",
        "SOURCE_AUTHORITY_READY",
    ],
    "value": [
        len(source_registry),
        (source_registry["source_class"] == "AUTHORITATIVE").sum(),
        (source_registry["source_class"] == "SCHEMA_REFERENCE").sum(),
        (source_registry["source_class"] == "REFERENCE_ONLY").sum(),
        len(missing_required),
        len(missing_authorities),
        len(invalid_reference_use),
        duplicate_source_ids,
        SOURCE_AUTHORITY_READY,
    ],
})

display(authority_summary)

assert SOURCE_AUTHORITY_READY, (
    "Section 1.1 failed. Review missing or incorrectly classified sources."
)

,item,value
0,Registered sources,9
1,Authoritative sources,4
2,Schema references,2
3,Reference-only sources,3
4,Missing required sources,0
5,Missing authority roles,0
6,Invalid reference usage,0
7,Duplicate source IDs,0
8,SOURCE_AUTHORITY_READY,True


# Section 1.2 — Source Fingerprint & Schema Drift Check

This section verifies that the registered project sources have not silently changed since setup.
Train features and labels are checked against their registered SHA256 fingerprints.
Tabular sources are also profiled by file size, row count, and ordered column signature.
The frozen fold manifest receives a stable content fingerprint for later lineage checks.
All transcript CSV files are hashed and combined into one deterministic directory fingerprint.
Transcript hashes use relative paths only, so the fingerprint is machine-independent.
Modification times are recorded for audit but are not used as source identity.
Detailed key, null, and dtype analysis is intentionally deferred to Section 1.3.
No transcript parsing or semantic processing is performed here.
The section ends with the `SOURCE_IDENTITY_READY` gate.

In [7]:
import hashlib

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)

    return digest.hexdigest()


def load_table(path):
    suffix = Path(path).suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(f"Unsupported tabular file: {path}")


def fingerprint_table(source_id, source_name, path, required, expected_hash=None, expected_rows=None):
    path = Path(path)

    if not path.exists():
        return {
            "source_id": source_id,
            "source_name": source_name,
            "required": required,
            "exists": False,
            "rows": None,
            "columns": None,
            "size_bytes": None,
            "modified_time": None,
            "sha256": None,
            "hash_status": "MISSING",
            "row_status": "MISSING",
            "column_signature": None,
            "passed": not required,
        }, None

    df = load_table(path)
    current_hash = sha256_file(path)
    column_text = "\x1f".join(map(str, df.columns))
    column_signature = hashlib.sha256(column_text.encode("utf-8")).hexdigest()

    hash_status = "BASELINE" if expected_hash is None else ("MATCH" if current_hash == expected_hash else "MISMATCH")
    row_status = "BASELINE" if expected_rows is None else ("MATCH" if len(df) == expected_rows else "MISMATCH")

    return {
        "source_id": source_id,
        "source_name": source_name,
        "required": required,
        "exists": True,
        "rows": len(df),
        "columns": len(df.columns),
        "size_bytes": path.stat().st_size,
        "modified_time": datetime.fromtimestamp(path.stat().st_mtime).astimezone().isoformat(),
        "sha256": current_hash,
        "hash_status": hash_status,
        "row_status": row_status,
        "column_signature": column_signature,
        "passed": hash_status != "MISMATCH" and row_status != "MISMATCH",
    }, df


expected_hashes = {
    "SRC_TRAIN_FEATURES": source_fingerprints.get("train_features_sha256"),
    "SRC_TRAIN_LABELS": source_fingerprints.get("train_labels_sha256"),
    "SRC_FROZEN_FOLDS": source_fingerprints.get("frozen_fold_manifest_sha256"),
}

expected_rows = {
    "SRC_TRAIN_FEATURES": setup_summary.get("response_count"),
    "SRC_TRAIN_LABELS": setup_summary.get("label_count"),
    "SRC_FROZEN_FOLDS": setup_summary.get("frozen_fold_rows"),
}

table_records = []
loaded_tables = {}

for row in source_registry.itertuples(index=False):
    if row.source_class == "REFERENCE_ONLY" or row.source_id == "SRC_TRANSCRIPTS" or row.path is None:
        continue

    record, df = fingerprint_table(
        row.source_id,
        row.source_name,
        row.path,
        row.required_now,
        expected_hashes.get(row.source_id),
        expected_rows.get(row.source_id),
    )

    table_records.append(record)

    if df is not None:
        loaded_tables[row.source_id] = df

table_fingerprint_audit = pd.DataFrame(table_records)

train_features = loaded_tables["SRC_TRAIN_FEATURES"]
train_labels = loaded_tables["SRC_TRAIN_LABELS"]
frozen_folds = loaded_tables["SRC_FROZEN_FOLDS"]

display(
    table_fingerprint_audit[
        ["source_name", "rows", "columns", "hash_status", "row_status", "passed"]
    ]
)

,source_name,rows,columns,hash_status,row_status,passed
0,Train features,35072,4,MATCH,BASELINE,True
1,Train labels,35072,2,MATCH,BASELINE,True
2,Frozen fold manifest,35072,3,BASELINE,BASELINE,True
3,Submission format 1,10508,2,BASELINE,BASELINE,True
4,Submission format 2,100,2,BASELINE,BASELINE,True


In [8]:
def fingerprint_transcript_directory(root):
    root = Path(root)
    files = sorted(root.rglob("*.csv"), key=lambda p: p.relative_to(root).as_posix())

    rows = []

    for path in files:
        relative_path = path.relative_to(root).as_posix()

        try:
            file_hash = sha256_file(path)
            error = None
        except Exception as e:
            file_hash = None
            error = str(e)

        rows.append({
            "relative_path": relative_path,
            "size_bytes": path.stat().st_size,
            "modified_time": datetime.fromtimestamp(path.stat().st_mtime).astimezone().isoformat(),
            "sha256": file_hash,
            "error": error,
        })

    manifest = pd.DataFrame(rows)
    failures = manifest["error"].notna().sum()

    if failures:
        return manifest, None

    digest = hashlib.sha256()

    for row in manifest.itertuples(index=False):
        line = f"{row.relative_path}\t{row.size_bytes}\t{row.sha256}\n"
        digest.update(line.encode("utf-8"))

    return manifest, digest.hexdigest()


assert path_registry["transcript_source_type"] == "external_path", (
    "Section 1.2 currently expects the registered external transcript directory."
)

TRANSCRIPT_ROOT = Path(path_registry["transcript_source"])

transcript_fingerprint_manifest, TRANSCRIPT_DIRECTORY_SHA256 = fingerprint_transcript_directory(TRANSCRIPT_ROOT)

transcript_file_count = len(transcript_fingerprint_manifest)
transcript_total_bytes = int(transcript_fingerprint_manifest["size_bytes"].sum())
transcript_hash_failures = int(transcript_fingerprint_manifest["error"].notna().sum())

registered_transcript_hash = source_fingerprints.get("transcript_directory_sha256")

if registered_transcript_hash is None:
    transcript_hash_status = "BASELINE"
else:
    transcript_hash_status = "MATCH" if TRANSCRIPT_DIRECTORY_SHA256 == registered_transcript_hash else "MISMATCH"

transcript_summary = pd.DataFrame({
    "item": [
        "Transcript files",
        "Total size MB",
        "Hash failures",
        "Directory hash status",
        "Directory SHA256",
    ],
    "value": [
        transcript_file_count,
        round(transcript_total_bytes / (1024 ** 2), 2),
        transcript_hash_failures,
        transcript_hash_status,
        TRANSCRIPT_DIRECTORY_SHA256,
    ],
})

display(transcript_summary)

,item,value
0,Transcript files,22821
1,Total size MB,573.02
2,Hash failures,0
3,Directory hash status,BASELINE
4,Directory SHA256,3f563b9911cd2f2e4d01dd5a457509ceaac6eb31ae8801...


In [9]:
transcript_record = {
    "source_id": "SRC_TRANSCRIPTS",
    "source_name": "Raw transcripts",
    "required": True,
    "exists": TRANSCRIPT_ROOT.exists(),
    "rows": None,
    "columns": None,
    "size_bytes": transcript_total_bytes,
    "modified_time": None,
    "sha256": TRANSCRIPT_DIRECTORY_SHA256,
    "hash_status": transcript_hash_status,
    "row_status": "NOT_CHECKED",
    "column_signature": None,
    "passed": (
        TRANSCRIPT_ROOT.exists()
        and transcript_file_count > 0
        and transcript_hash_failures == 0
        and transcript_hash_status != "MISMATCH"
    ),
}

source_identity_audit = pd.concat(
    [table_fingerprint_audit, pd.DataFrame([transcript_record])],
    ignore_index=True,
)

current_source_fingerprints = {
    "train_features_sha256": source_identity_audit.loc[
        source_identity_audit["source_id"] == "SRC_TRAIN_FEATURES", "sha256"
    ].iloc[0],
    "train_labels_sha256": source_identity_audit.loc[
        source_identity_audit["source_id"] == "SRC_TRAIN_LABELS", "sha256"
    ].iloc[0],
    "frozen_fold_manifest_sha256": source_identity_audit.loc[
        source_identity_audit["source_id"] == "SRC_FROZEN_FOLDS", "sha256"
    ].iloc[0],
    "transcript_directory_sha256": TRANSCRIPT_DIRECTORY_SHA256,
    "transcript_file_count": transcript_file_count,
    "transcript_total_bytes": transcript_total_bytes,
}

required_failures = source_identity_audit[
    source_identity_audit["required"] & ~source_identity_audit["passed"]
]

SOURCE_IDENTITY_READY = required_failures.empty

identity_summary = pd.DataFrame({
    "item": [
        "Checked tabular sources",
        "Transcript files",
        "Transcript hash failures",
        "Required source failures",
        "Train features hash",
        "Train labels hash",
        "Frozen fold hash",
        "Transcript directory hash",
        "SOURCE_IDENTITY_READY",
    ],
    "value": [
        len(table_fingerprint_audit),
        transcript_file_count,
        transcript_hash_failures,
        len(required_failures),
        table_fingerprint_audit.loc[
            table_fingerprint_audit["source_id"] == "SRC_TRAIN_FEATURES", "hash_status"
        ].iloc[0],
        table_fingerprint_audit.loc[
            table_fingerprint_audit["source_id"] == "SRC_TRAIN_LABELS", "hash_status"
        ].iloc[0],
        table_fingerprint_audit.loc[
            table_fingerprint_audit["source_id"] == "SRC_FROZEN_FOLDS", "hash_status"
        ].iloc[0],
        transcript_hash_status,
        SOURCE_IDENTITY_READY,
    ],
})

display(identity_summary)

assert SOURCE_IDENTITY_READY, (
    "Section 1.2 failed.\n\n"
    + required_failures[
        ["source_name", "hash_status", "row_status"]
    ].to_string(index=False)
)

,item,value
0,Checked tabular sources,5
1,Transcript files,22821
2,Transcript hash failures,0
3,Required source failures,0
4,Train features hash,MATCH
5,Train labels hash,MATCH
6,Frozen fold hash,BASELINE
7,Transcript directory hash,BASELINE
8,SOURCE_IDENTITY_READY,True


# Section 1.3 — Schema Discovery & Key Resolution

This section establishes the exact schema and key contract of the authoritative tabular sources.
Each column is profiled for dtype, nulls, uniqueness, duplicates, blank values, and whitespace issues.
`response_id` is validated as the response key and must align exactly between features and labels.
`session_id` is validated as a non-null repeated session key.
Learning-objective text must be present and non-empty, while objective-ID mappings are audited separately.
The target field must contain only valid binary values.
Response and session key dtypes are checked across features, labels, and frozen folds.
Objective ID-to-text and text-to-ID inconsistencies are detected but not silently repaired.
Full objective reconciliation is deferred to Section 1.6 and full fold validation to Section 1.9.
The section ends with the `SCHEMA_KEYS_VALID` gate.

In [10]:
from pandas.api.types import is_string_dtype, is_integer_dtype, is_float_dtype, is_bool_dtype

assert SOURCE_IDENTITY_READY, "Section 1.2 must pass before Section 1.3."

def dtype_family(series):
    if is_bool_dtype(series): return "boolean"
    if is_integer_dtype(series): return "integer"
    if is_float_dtype(series): return "float"
    if is_string_dtype(series): return "string"
    return str(series.dtype)

def text_issue_counts(series):
    text = series.astype("string")
    valid = text.notna()

    blank = (valid & text.str.strip().eq("")).sum()
    whitespace = (valid & text.ne(text.str.strip())).sum()

    return int(blank), int(whitespace)

def profile_dataframe(name, df):
    rows = []

    for column in df.columns:
        series = df[column]
        blank_count, whitespace_count = text_issue_counts(series)

        rows.append({
            "source": name,
            "column": column,
            "dtype": str(series.dtype),
            "dtype_family": dtype_family(series),
            "rows": len(series),
            "null_count": int(series.isna().sum()),
            "null_rate": round(series.isna().mean(), 6),
            "unique_count": int(series.nunique(dropna=True)),
            "duplicate_count": int(series.duplicated().sum()),
            "blank_count": blank_count,
            "whitespace_count": whitespace_count,
            "candidate_key": series.notna().all() and series.nunique(dropna=True) == len(series),
        })

    return pd.DataFrame(rows)

schema_profile = pd.concat([
    profile_dataframe("train_features", train_features),
    profile_dataframe("train_labels", train_labels),
    profile_dataframe("frozen_folds", frozen_folds),
], ignore_index=True)

display(schema_profile)

,source,column,dtype,dtype_family,rows,null_count,null_rate,unique_count,duplicate_count,blank_count,whitespace_count,candidate_key
0,train_features,response_id,object,string,35072,0,0.0,35072,0,0,0,True
1,train_features,session_id,object,string,35072,0,0.0,22821,12251,0,0,False
2,train_features,learning_objective_id,object,string,35072,0,0.0,398,34674,0,0,False
3,train_features,learning_objective,object,string,35072,0,0.0,398,34674,0,3809,False
4,train_labels,response_id,object,string,35072,0,0.0,35072,0,0,0,True
5,train_labels,is_correct,float64,float,35072,0,0.0,2,35070,0,0,False
6,frozen_folds,response_id,object,string,35072,0,0.0,35072,0,0,0,True
7,frozen_folds,session_id,object,string,35072,0,0.0,22821,12251,0,0,False
8,frozen_folds,fold,int64,integer,35072,0,0.0,5,35067,0,0,False


In [13]:
required_columns = {
    "train_features": {"response_id", "session_id", "learning_objective_id", "learning_objective"},
    "train_labels": {"response_id", "is_correct"},
    "frozen_folds": {"response_id", "session_id", "fold"},
}

frames = {
    "train_features": train_features,
    "train_labels": train_labels,
    "frozen_folds": frozen_folds,
}

missing_columns = {
    name: sorted(columns - set(frames[name].columns))
    for name, columns in required_columns.items()
}

assert not any(missing_columns.values()), f"Missing required columns: {missing_columns}"

RESOLVED_SCHEMA = {
    "response_key": "response_id",
    "session_key": "session_id",
    "objective_id_field": "learning_objective_id",
    "objective_field": "learning_objective",
    "target_field": "is_correct",
    "fold_field": "fold",
}

feature_ids = set(train_features["response_id"])
label_ids = set(train_labels["response_id"])

features_only_ids = feature_ids - label_ids
labels_only_ids = label_ids - feature_ids

target_values = set(train_labels["is_correct"].dropna().unique())
valid_target_domain = target_values.issubset({0, 1})

response_dtype_ok = (
    dtype_family(train_features["response_id"])
    == dtype_family(train_labels["response_id"])
    == dtype_family(frozen_folds["response_id"])
)

session_dtype_ok = (
    dtype_family(train_features["session_id"])
    == dtype_family(frozen_folds["session_id"])
)

objective_id_to_text = (
    train_features.groupby("learning_objective_id")["learning_objective"]
    .nunique(dropna=False)
)

objective_text_to_id = (
    train_features.groupby("learning_objective")["learning_objective_id"]
    .nunique(dropna=False)
)

objective_id_to_text_conflicts = objective_id_to_text[objective_id_to_text > 1]
objective_text_to_id_conflicts = objective_text_to_id[objective_text_to_id > 1]

def check_row(check, passed, detail, required=True):
    return {
        "check": check,
        "required": required,
        "passed": bool(passed),
        "detail": detail,
    }

schema_checks = pd.DataFrame([
    check_row("Feature response_id unique", train_features["response_id"].is_unique, train_features["response_id"].duplicated().sum()),
    check_row("Label response_id unique", train_labels["response_id"].is_unique, train_labels["response_id"].duplicated().sum()),
    check_row("Feature response_id non-null", train_features["response_id"].notna().all(), train_features["response_id"].isna().sum()),
    check_row("Label response_id non-null", train_labels["response_id"].notna().all(), train_labels["response_id"].isna().sum()),
    check_row("Feature-label response IDs aligned", not features_only_ids and not labels_only_ids, f"features_only={len(features_only_ids)}, labels_only={len(labels_only_ids)}"),
    check_row("Session IDs non-null", train_features["session_id"].notna().all(), train_features["session_id"].isna().sum()),
    check_row("Session IDs non-blank", text_issue_counts(train_features["session_id"])[0] == 0, text_issue_counts(train_features["session_id"])[0]),
    check_row("Objective text non-null", train_features["learning_objective"].notna().all(), train_features["learning_objective"].isna().sum()),
    check_row("Objective text non-blank", text_issue_counts(train_features["learning_objective"])[0] == 0, text_issue_counts(train_features["learning_objective"])[0]),
    check_row("Target non-null", train_labels["is_correct"].notna().all(), train_labels["is_correct"].isna().sum()),
    check_row("Target domain valid", valid_target_domain, sorted(target_values)),
    check_row("Fold non-null", frozen_folds["fold"].notna().all(), frozen_folds["fold"].isna().sum()),
    check_row("Response key dtype compatible", response_dtype_ok, f"{dtype_family(train_features['response_id'])} / {dtype_family(train_labels['response_id'])} / {dtype_family(frozen_folds['response_id'])}"),
    check_row("Session key dtype compatible", session_dtype_ok, f"{dtype_family(train_features['session_id'])} / {dtype_family(frozen_folds['session_id'])}"),
    check_row("Objective ID non-null", train_features["learning_objective_id"].notna().all(), train_features["learning_objective_id"].isna().sum(), required=False),
    check_row("Objective ID → text mapping", objective_id_to_text_conflicts.empty, len(objective_id_to_text_conflicts), required=False),
    check_row("Objective text → ID mapping", objective_text_to_id_conflicts.empty, len(objective_text_to_id_conflicts), required=False),
    check_row("Objective text boundary whitespace",text_issue_counts(train_features["learning_objective"])[1] == 0,text_issue_counts(train_features["learning_objective"])[1],required=False,
),
])


objective_mapping_summary = pd.DataFrame({
    "item": [
        "Unique objective IDs",
        "Unique exact objective texts",
        "IDs mapped to multiple texts",
        "Texts mapped to multiple IDs",
    ],
    "value": [
        train_features["learning_objective_id"].nunique(dropna=True),
        train_features["learning_objective"].nunique(dropna=True),
        len(objective_id_to_text_conflicts),
        len(objective_text_to_id_conflicts),
    ],
})

display(schema_checks)
display(objective_mapping_summary)

,check,required,passed,detail
0,Feature response_id unique,True,True,0
1,Label response_id unique,True,True,0
2,Feature response_id non-null,True,True,0
3,Label response_id non-null,True,True,0
4,Feature-label response IDs aligned,True,True,"features_only=0, labels_only=0"
5,Session IDs non-null,True,True,0
6,Session IDs non-blank,True,True,0
7,Objective text non-null,True,True,0
8,Objective text non-blank,True,True,0
9,Target non-null,True,True,0


,item,value
0,Unique objective IDs,398
1,Unique exact objective texts,398
2,IDs mapped to multiple texts,0
3,Texts mapped to multiple IDs,0


In [14]:
required_failures = schema_checks[
    schema_checks["required"] & ~schema_checks["passed"]
]

schema_warnings = schema_checks[
    ~schema_checks["required"] & ~schema_checks["passed"]
]

SCHEMA_KEYS_VALID = required_failures.empty

schema_summary = pd.DataFrame({
    "item": [
        "Feature rows",
        "Label rows",
        "Feature response duplicates",
        "Label response duplicates",
        "Feature-only response IDs",
        "Label-only response IDs",
        "Objective ID mapping issues",
        "Objective text mapping issues",
        "Required failures",
        "Non-blocking warnings",
        "SCHEMA_KEYS_VALID",
    ],
    "value": [
        len(train_features),
        len(train_labels),
        int(train_features["response_id"].duplicated().sum()),
        int(train_labels["response_id"].duplicated().sum()),
        len(features_only_ids),
        len(labels_only_ids),
        len(objective_id_to_text_conflicts),
        len(objective_text_to_id_conflicts),
        len(required_failures),
        len(schema_warnings),
        SCHEMA_KEYS_VALID,
    ],
})

resolved_schema_table = pd.DataFrame(
    RESOLVED_SCHEMA.items(),
    columns=["role", "column"]
)

display(resolved_schema_table)
display(schema_summary)

assert SCHEMA_KEYS_VALID, (
    "Section 1.3 failed.\n\n"
    + required_failures[["check", "detail"]].to_string(index=False)
)

,role,column
0,response_key,response_id
1,session_key,session_id
2,objective_id_field,learning_objective_id
3,objective_field,learning_objective
4,target_field,is_correct
5,fold_field,fold


,item,value
0,Feature rows,35072
1,Label rows,35072
2,Feature response duplicates,0
3,Label response duplicates,0
4,Feature-only response IDs,0
5,Label-only response IDs,0
6,Objective ID mapping issues,0
7,Objective text mapping issues,0
8,Required failures,0
9,Non-blocking warnings,1


# Section 1.4 — Build Response Foundation

This section creates the official response-level foundation used by the rest of Phase 1.
Training features provide response, session, and learning-objective metadata.
Training labels are joined strictly by `response_id`.
Frozen folds are attached using both `response_id` and `session_id`.
Every merge uses explicit one-to-one validation to prevent row multiplication or silent loss.
Raw objective text and the original objective ID are preserved without normalization.
No transcript data, objective prior, retrieval score, or model feature is added here.
The final unit is one row per session–objective response.
The section saves `responses_base.parquet` only after all integrity checks pass.
The section ends with the `RESPONSE_FOUNDATION_READY` gate.

In [15]:
assert SCHEMA_KEYS_VALID, "Section 1.3 must pass before Section 1.4."

features_base = train_features[
    ["response_id", "session_id", "learning_objective_id", "learning_objective"]
].rename(columns={
    "learning_objective_id": "objective_id_raw",
    "learning_objective": "objective_raw",
}).copy()

labels_base = train_labels[
    ["response_id", "is_correct"]
].rename(columns={
    "is_correct": "target",
}).copy()

folds_base = frozen_folds[
    ["response_id", "session_id", "fold"]
].copy()

source_summary = pd.DataFrame({
    "source": ["features_base", "labels_base", "folds_base"],
    "rows": [len(features_base), len(labels_base), len(folds_base)],
    "columns": [len(features_base.columns), len(labels_base.columns), len(folds_base.columns)],
    "response_unique": [
        features_base["response_id"].is_unique,
        labels_base["response_id"].is_unique,
        folds_base["response_id"].is_unique,
    ],
})

display(source_summary)

,source,rows,columns,response_unique
0,features_base,35072,4,True
1,labels_base,35072,2,True
2,folds_base,35072,3,True


In [16]:
def merge_audit(stage, left_rows, right_rows, result_rows, unmatched, status):
    return {
        "stage": stage,
        "left_rows": left_rows,
        "right_rows": right_rows,
        "result_rows": result_rows,
        "unmatched": unmatched,
        "status": status,
    }

label_unmatched = features_base["response_id"].isin(labels_base["response_id"]).eq(False).sum()

responses_labeled = features_base.merge(
    labels_base,
    on="response_id",
    how="left",
    validate="one_to_one",
)

label_join_ok = (
    len(responses_labeled) == len(features_base)
    and responses_labeled["target"].notna().all()
    and label_unmatched == 0
)

fold_keys = ["response_id", "session_id"]

fold_unmatched = (
    responses_labeled[fold_keys]
    .merge(folds_base[fold_keys], on=fold_keys, how="left", indicator=True)
    ["_merge"]
    .ne("both")
    .sum()
)

responses_base = responses_labeled.merge(
    folds_base,
    on=fold_keys,
    how="left",
    validate="one_to_one",
)

fold_join_ok = (
    len(responses_base) == len(responses_labeled)
    and responses_base["fold"].notna().all()
    and fold_unmatched == 0
)

response_join_audit = pd.DataFrame([
    merge_audit(
        "Features + Labels",
        len(features_base),
        len(labels_base),
        len(responses_labeled),
        int(label_unmatched),
        "PASS" if label_join_ok else "FAIL",
    ),
    merge_audit(
        "Responses + Frozen Folds",
        len(responses_labeled),
        len(folds_base),
        len(responses_base),
        int(fold_unmatched),
        "PASS" if fold_join_ok else "FAIL",
    ),
])

responses_base = responses_base[
    ["response_id", "session_id", "objective_id_raw", "objective_raw", "target", "fold"]
]

display(response_join_audit)
display(responses_base.head())

,stage,left_rows,right_rows,result_rows,unmatched,status
0,Features + Labels,35072,35072,35072,0,PASS
1,Responses + Frozen Folds,35072,35072,35072,0,PASS


,response_id,session_id,objective_id_raw,objective_raw,target,fold
0,aaaavsh,bcaufvc,dqibnvd,Knowing the value of each digit in numbers wit...,1.0,0
1,aaabhzi,eyutanf,eukmzxl,Adding and subtracting tens to a 2-digit number.,1.0,4
2,aaahpnz,juptkxd,fjbqcsv,Comparing and ordering fractions by finding a ...,0.0,2
3,aaajpom,ntwkcfj,acvbcev,Comparing fractions using reasoning.,0.0,0
4,aaamwux,jqriibm,krfuudx,Counting in multiples.,0.0,0


In [17]:
def count_blank(series):
    text = series.astype("string")
    return int((text.notna() & text.str.strip().eq("")).sum())

target_values = set(responses_base["target"].dropna().unique())

response_checks = pd.DataFrame([
    check_row("Row count preserved", len(responses_base) == len(features_base), f"{len(responses_base)} / {len(features_base)}"),
    check_row("response_id unique", responses_base["response_id"].is_unique, responses_base["response_id"].duplicated().sum()),
    check_row("response_id non-null", responses_base["response_id"].notna().all(), responses_base["response_id"].isna().sum()),
    check_row("session_id non-null", responses_base["session_id"].notna().all(), responses_base["session_id"].isna().sum()),
    check_row("objective_raw non-null", responses_base["objective_raw"].notna().all(), responses_base["objective_raw"].isna().sum()),
    check_row("objective_raw non-blank", count_blank(responses_base["objective_raw"]) == 0, count_blank(responses_base["objective_raw"])),
    check_row("target non-null", responses_base["target"].notna().all(), responses_base["target"].isna().sum()),
    check_row("target domain valid", target_values.issubset({0, 1}), sorted(target_values)),
    check_row("fold non-null", responses_base["fold"].notna().all(), responses_base["fold"].isna().sum()),
    check_row("Label join valid", label_join_ok, response_join_audit.loc[0, "status"]),
    check_row("Fold join valid", fold_join_ok, response_join_audit.loc[1, "status"]),
    check_row(
        "Objective boundary whitespace",
        text_issue_counts(responses_base["objective_raw"])[1] == 0,
        text_issue_counts(responses_base["objective_raw"])[1],
        required=False,
    ),
])

required_failures = response_checks[
    response_checks["required"] & ~response_checks["passed"]
]

response_warnings = response_checks[
    ~response_checks["required"] & ~response_checks["passed"]
]

RESPONSE_FOUNDATION_READY = required_failures.empty

response_summary = pd.DataFrame({
    "item": [
        "Response rows",
        "Unique response IDs",
        "Unique sessions",
        "Unique objective IDs",
        "Unique raw objective texts",
        "Missing targets",
        "Missing folds",
        "Required failures",
        "Non-blocking warnings",
        "RESPONSE_FOUNDATION_READY",
    ],
    "value": [
        len(responses_base),
        responses_base["response_id"].nunique(),
        responses_base["session_id"].nunique(),
        responses_base["objective_id_raw"].nunique(),
        responses_base["objective_raw"].nunique(),
        responses_base["target"].isna().sum(),
        responses_base["fold"].isna().sum(),
        len(required_failures),
        len(response_warnings),
        RESPONSE_FOUNDATION_READY,
    ],
})

display(response_checks)
display(response_summary)

assert RESPONSE_FOUNDATION_READY, (
    "Section 1.4 failed.\n\n"
    + required_failures[["check", "detail"]].to_string(index=False)
)

RESPONSES_BASE_PATH = INVENTORY_OUTPUT_DIR / "responses_base.parquet"
responses_base.to_parquet(RESPONSES_BASE_PATH, index=False)

print(f"Saved: {RESPONSES_BASE_PATH}")

,check,required,passed,detail
0,Row count preserved,True,True,35072 / 35072
1,response_id unique,True,True,0
2,response_id non-null,True,True,0
3,session_id non-null,True,True,0
4,objective_raw non-null,True,True,0
5,objective_raw non-blank,True,True,0
6,target non-null,True,True,0
7,target domain valid,True,True,"[0.0, 1.0]"
8,fold non-null,True,True,0
9,Label join valid,True,True,PASS


,item,value
0,Response rows,35072
1,Unique response IDs,35072
2,Unique sessions,22821
3,Unique objective IDs,398
4,Unique raw objective texts,398
5,Missing targets,0
6,Missing folds,0
7,Required failures,0
8,Non-blocking warnings,1
9,RESPONSE_FOUNDATION_READY,True


Saved: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\01_data_foundation\01_inventory\responses_base.parquet


# Section 1.5 — Population Census

This section documents the exact population represented by `responses_base`.
It counts responses, sessions, objectives, positive rows, negative rows, and the global target rate.
The session–objective response unit is validated for structural uniqueness.
Responses per session and unique objectives per session are summarized without modelling interpretation.
A temporary safe objective preview is created using whitespace-only normalization.
Raw objective text remains unchanged and continues to be the authoritative source value.
The safe preview is used only to detect possible normalization-related count changes.
Historical objective-count differences are not resolved in this section.
No objective-level target statistics, transcript analysis, or model features are created.
The section ends with the `POPULATION_CENSUS_READY` gate.

In [18]:
assert RESPONSE_FOUNDATION_READY, "Section 1.4 must pass before Section 1.5."

n_responses = len(responses_base)
n_sessions = responses_base["session_id"].nunique()
n_objective_ids = responses_base["objective_id_raw"].nunique()
n_objectives_raw = responses_base["objective_raw"].nunique()

n_positive = int((responses_base["target"] == 1).sum())
n_negative = int((responses_base["target"] == 0).sum())
positive_rate = n_positive / n_responses
negative_rate = n_negative / n_responses

population_summary = pd.DataFrame({
    "item": [
        "Responses",
        "Sessions",
        "Objective IDs",
        "Raw objective texts",
        "Positive rows",
        "Negative rows",
        "Positive rate",
        "Negative rate",
    ],
    "value": [
        n_responses,
        n_sessions,
        n_objective_ids,
        n_objectives_raw,
        n_positive,
        n_negative,
        round(positive_rate, 6),
        round(negative_rate, 6),
    ],
})

setup_reference = pd.DataFrame({
    "metric": ["Responses", "Sessions", "Objectives", "Positive rate"],
    "current": [n_responses, n_sessions, n_objectives_raw, round(positive_rate, 6)],
    "setup_reference": [
        setup_summary.get("response_count"),
        setup_summary.get("session_count"),
        setup_summary.get("objective_count"),
        round(setup_summary.get("positive_rate"), 6) if setup_summary.get("positive_rate") is not None else None,
    ],
})

setup_reference["match"] = setup_reference["current"].eq(setup_reference["setup_reference"])

display(population_summary)
display(setup_reference)

,item,value
0,Responses,35072.000000
1,Sessions,22821.000000
2,Objective IDs,398.000000
3,Raw objective texts,398.000000
4,Positive rows,24637.000000
5,Negative rows,10435.000000
6,Positive rate,0.702469
7,Negative rate,0.297531


,metric,current,setup_reference,match
0,Responses,35072.000000,NaN,False
1,Sessions,22821.000000,22821.000000,True
2,Objectives,398.000000,398.000000,True
3,Positive rate,0.702469,0.702469,True


In [19]:
import re
import unicodedata

def safe_objective_preview(text):
    if pd.isna(text):
        return pd.NA

    text = unicodedata.normalize("NFKC", str(text))
    return re.sub(r"\s+", " ", text).strip()

objective_safe_preview = responses_base["objective_raw"].map(safe_objective_preview)

session_population = (
    responses_base.assign(objective_safe_preview=objective_safe_preview)
    .groupby("session_id")
    .agg(
        response_count=("response_id", "size"),
        objective_id_count=("objective_id_raw", "nunique"),
        objective_raw_count=("objective_raw", "nunique"),
        objective_safe_count=("objective_safe_preview", "nunique"),
    )
    .reset_index()
)

duplicate_session_objective_ids = int(
    responses_base.duplicated(["session_id", "objective_id_raw"]).sum()
)

duplicate_session_objective_texts = int(
    responses_base.duplicated(["session_id", "objective_raw"]).sum()
)

safe_pair_table = responses_base[["session_id"]].copy()
safe_pair_table["objective_safe_preview"] = objective_safe_preview

duplicate_safe_pairs = int(
    safe_pair_table.duplicated(["session_id", "objective_safe_preview"]).sum()
)

session_structure_mismatch = int(
    (
        (session_population["response_count"] != session_population["objective_id_count"]) |
        (session_population["response_count"] != session_population["objective_raw_count"])
    ).sum()
)

n_objectives_safe_preview = objective_safe_preview.nunique()

session_size_distribution = (
    session_population["response_count"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95])
    .rename("responses_per_session")
    .to_frame()
)

session_frequency = (
    session_population["response_count"]
    .value_counts()
    .sort_index()
    .rename_axis("responses_per_session")
    .reset_index(name="session_count")
)

objective_preview_summary = pd.DataFrame({
    "item": [
        "Raw objective texts",
        "Safe-preview objective texts",
        "Raw → safe count difference",
        "Duplicate session-objective ID pairs",
        "Duplicate session-objective raw-text pairs",
        "Duplicate session-objective safe-preview pairs",
        "Sessions with structural mismatch",
    ],
    "value": [
        n_objectives_raw,
        n_objectives_safe_preview,
        n_objectives_raw - n_objectives_safe_preview,
        duplicate_session_objective_ids,
        duplicate_session_objective_texts,
        duplicate_safe_pairs,
        session_structure_mismatch,
    ],
})

display(session_size_distribution)
display(session_frequency)
display(objective_preview_summary)

,responses_per_session
count,22821.000000
mean,1.536830
std,0.857158
min,1.000000
50%,1.000000
75%,2.000000
90%,3.000000
95%,3.000000
max,10.000000


,responses_per_session,session_count
0,1,14457
1,2,5640
2,3,1904
3,4,581
4,5,168
5,6,48
6,7,17
7,8,4
8,10,2


,item,value
0,Raw objective texts,398
1,Safe-preview objective texts,398
2,Raw → safe count difference,0
3,Duplicate session-objective ID pairs,0
4,Duplicate session-objective raw-text pairs,0
5,Duplicate session-objective safe-preview pairs,0
6,Sessions with structural mismatch,0


In [20]:
population_checks = pd.DataFrame([
    check_row(
        "Target accounting complete",
        n_positive + n_negative == n_responses,
        f"{n_positive} + {n_negative} = {n_positive + n_negative}"
    ),
    check_row(
        "Session-objective ID pairs unique",
        duplicate_session_objective_ids == 0,
        duplicate_session_objective_ids
    ),
    check_row(
        "Session-objective raw-text pairs unique",
        duplicate_session_objective_texts == 0,
        duplicate_session_objective_texts
    ),
    check_row(
        "Session structure internally consistent",
        session_structure_mismatch == 0,
        session_structure_mismatch
    ),
    check_row(
        "Every session has at least one response",
        session_population["response_count"].ge(1).all(),
        int(session_population["response_count"].lt(1).sum())
    ),
    check_row(
        "Every session has an objective",
        session_population["objective_id_count"].ge(1).all(),
        int(session_population["objective_id_count"].lt(1).sum())
    ),
    check_row(
        "Setup response count matches",
        setup_summary.get("response_count") in {None, n_responses},
        f"current={n_responses}, setup={setup_summary.get('response_count')}",
        required=False
    ),
    check_row(
        "Setup session count matches",
        setup_summary.get("session_count") in {None, n_sessions},
        f"current={n_sessions}, setup={setup_summary.get('session_count')}",
        required=False
    ),
    check_row(
        "Raw and safe-preview objective counts match",
        n_objectives_raw == n_objectives_safe_preview,
        f"raw={n_objectives_raw}, safe={n_objectives_safe_preview}",
        required=False
    ),
    check_row(
        "No duplicate safe-preview session-objective pairs",
        duplicate_safe_pairs == 0,
        duplicate_safe_pairs,
        required=False
    ),
])

population_failures = population_checks[
    population_checks["required"] & ~population_checks["passed"]
]

population_warnings = population_checks[
    ~population_checks["required"] & ~population_checks["passed"]
]

POPULATION_CENSUS_READY = population_failures.empty

population_gate_summary = pd.DataFrame({
    "item": [
        "Responses",
        "Sessions",
        "Raw objectives",
        "Safe-preview objectives",
        "Positive rows",
        "Negative rows",
        "Duplicate raw session-objective pairs",
        "Structural mismatch sessions",
        "Required failures",
        "Non-blocking warnings",
        "POPULATION_CENSUS_READY",
    ],
    "value": [
        n_responses,
        n_sessions,
        n_objectives_raw,
        n_objectives_safe_preview,
        n_positive,
        n_negative,
        duplicate_session_objective_texts,
        session_structure_mismatch,
        len(population_failures),
        len(population_warnings),
        POPULATION_CENSUS_READY,
    ],
})

display(population_checks)
display(population_gate_summary)

assert POPULATION_CENSUS_READY, (
    "Section 1.5 failed.\n\n"
    + population_failures[["check", "detail"]].to_string(index=False)
)

,check,required,passed,detail
0,Target accounting complete,True,True,24637 + 10435 = 35072
1,Session-objective ID pairs unique,True,True,0
2,Session-objective raw-text pairs unique,True,True,0
3,Session structure internally consistent,True,True,0
4,Every session has at least one response,True,True,0
5,Every session has an objective,True,True,0
6,Setup response count matches,False,True,"current=35072, setup=None"
7,Setup session count matches,False,True,"current=22821, setup=22821"
8,Raw and safe-preview objective counts match,False,True,"raw=398, safe=398"
9,No duplicate safe-preview session-objective pairs,False,True,0


,item,value
0,Responses,35072
1,Sessions,22821
2,Raw objectives,398
3,Safe-preview objectives,398
4,Positive rows,24637
5,Negative rows,10435
6,Duplicate raw session-objective pairs,0
7,Structural mismatch sessions,0
8,Required failures,0
9,Non-blocking warnings,0


# Section 1.6 — Objective Population Reconciliation

This section certifies the current learning-objective population and investigates the historical 398-vs-396 difference.
Current objective IDs, raw texts, safe-normalized texts, and case-normalized diagnostic keys are compared.
Response, session, and fold coverage are summarized for every current objective.
The current official training features remain the authoritative source of objective identity.
Historical baseline artifacts are used only as reference evidence and never redefine the current population.
A historical artifact is searched only if it contains response-level objective information.
Any current objectives collapsed into one historical identity are reported explicitly.
No target rates, objective priors, semantic clustering, or model features are created here.
Historical reconciliation may remain a documented warning if the old artifact is unavailable.
The section ends with the `OBJECTIVE_RECONCILIATION_READY` gate.

In [21]:
assert POPULATION_CENSUS_READY, "Section 1.5 must pass before Section 1.6."

def case_key(text):
    return safe_objective_preview(text).casefold() if pd.notna(text) else pd.NA

objective_work = responses_base[
    ["response_id", "session_id", "objective_id_raw", "objective_raw", "fold"]
].copy()

objective_work["objective_safe_norm"] = objective_work["objective_raw"].map(safe_objective_preview)
objective_work["objective_case_key"] = objective_work["objective_safe_norm"].map(case_key)

def fold_list(series):
    return ",".join(map(str, sorted(series.dropna().astype(int).unique())))

current_objective_catalogue = (
    objective_work.groupby("objective_id_raw", as_index=False)
    .agg(
        objective_raw=("objective_raw", "first"),
        objective_safe_norm=("objective_safe_norm", "first"),
        objective_case_key=("objective_case_key", "first"),
        raw_text_variants=("objective_raw", "nunique"),
        safe_text_variants=("objective_safe_norm", "nunique"),
        response_count=("response_id", "size"),
        session_count=("session_id", "nunique"),
        fold_count=("fold", "nunique"),
        folds_present=("fold", fold_list),
    )
)

raw_to_id_conflicts = (
    objective_work.groupby("objective_raw")["objective_id_raw"]
    .nunique()
    .gt(1)
    .sum()
)

safe_to_id_conflicts = (
    objective_work.groupby("objective_safe_norm")["objective_id_raw"]
    .nunique()
    .gt(1)
    .sum()
)

identity_summary = pd.DataFrame({
    "identity_level": [
        "Objective IDs",
        "Raw objective texts",
        "Safe-normalized texts",
        "Case-normalized keys",
        "Objectives with fold coverage",
        "Objectives present in all 5 folds",
    ],
    "count": [
        objective_work["objective_id_raw"].nunique(),
        objective_work["objective_raw"].nunique(),
        objective_work["objective_safe_norm"].nunique(),
        objective_work["objective_case_key"].nunique(),
        int(current_objective_catalogue["fold_count"].gt(0).sum()),
        int(current_objective_catalogue["fold_count"].eq(5).sum()),
    ],
})

display(identity_summary)
display(current_objective_catalogue.head())

,identity_level,count
0,Objective IDs,398
1,Raw objective texts,398
2,Safe-normalized texts,398
3,Case-normalized keys,396
4,Objectives with fold coverage,398
5,Objectives present in all 5 folds,190


,objective_id_raw,objective_raw,objective_safe_norm,objective_case_key,raw_text_variants,safe_text_variants,response_count,session_count,fold_count,folds_present
0,aabhqcm,Finding prime factors,Finding prime factors,finding prime factors,1,1,1,1,1,1
1,abjzaky,"Measuring and comparing length, mass and capacity","Measuring and comparing length, mass and capacity","measuring and comparing length, mass and capacity",1,1,1,1,1,4
2,acvbcev,Comparing fractions using reasoning.,Comparing fractions using reasoning.,comparing fractions using reasoning.,1,1,510,510,5,"0,1,2,3,4"
3,advgudi,Using a number line to show fractions.,Using a number line to show fractions.,using a number line to show fractions.,1,1,112,112,5,"0,1,2,3,4"
4,aetwckq,Sharing in ratio.,Sharing in ratio.,sharing in ratio.,1,1,7,7,4,"1,2,3,4"


In [22]:
def reference_search_roots():
    roots = []

    for key, value in path_registry.items():
        if not isinstance(value, str):
            continue

        text = f"{key} {value}".lower()

        if any(token in text for token in ["baseline", "oof", "master", "analysis"]):
            path = Path(value)

            if path.exists():
                roots.append(path if path.is_dir() else path.parent)

    for path in [
        Path(path_registry["data_root"]) / "outputs",
        PROJECT_ROOT / "outputs",
    ]:
        if path.exists():
            roots.append(path)

    unique = []

    for path in roots:
        path = path.resolve()

        if path not in unique:
            unique.append(path)

    return unique


def objective_columns(columns):
    preferred = [
        "learning_objective",
        "objective_raw",
        "objective",
        "learning_objective_id",
        "objective_id",
        "objective_text",
    ]

    found = [column for column in preferred if column in columns]
    found += [
        column for column in columns
        if "objective" in column.lower() and column not in found
    ]

    return found


def inspect_reference_file(path):
    try:
        if path.suffix.lower() == ".csv":
            header = pd.read_csv(path, nrows=0)
        else:
            header = pd.read_parquet(path).head(0)

        if "response_id" not in header.columns:
            return []

        objective_cols = objective_columns(header.columns)

        if not objective_cols:
            return []

        usecols = ["response_id", *objective_cols]

        if path.suffix.lower() == ".csv":
            data = pd.read_csv(path, usecols=usecols)
        else:
            data = pd.read_parquet(path, columns=usecols)

        records = []

        for column in objective_cols:
            records.append({
                "path": str(path),
                "rows": len(data),
                "response_unique": data["response_id"].is_unique,
                "objective_column": column,
                "objective_count": data[column].nunique(dropna=True),
            })

        return records

    except Exception:
        return []


search_tokens = ["oof", "prediction", "validation", "baseline", "master"]
reference_paths = set()

for root in reference_search_roots():
    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in {".csv", ".parquet"}:
            continue

        if TRANSCRIPT_ROOT in path.parents or PHASE1_ROOT in path.parents:
            continue

        relative_text = str(path).lower()

        if any(token in relative_text for token in search_tokens):
            reference_paths.add(path)

historical_records = []

for path in sorted(reference_paths):
    historical_records.extend(inspect_reference_file(path))

historical_candidate_summary = pd.DataFrame(historical_records)

if not historical_candidate_summary.empty:
    historical_candidate_summary["population_match"] = historical_candidate_summary["rows"].eq(n_responses)
    historical_candidate_summary["historical_396_match"] = historical_candidate_summary["objective_count"].eq(396)
    historical_candidate_summary["oof_hint"] = historical_candidate_summary["path"].str.lower().str.contains("oof")

    historical_candidate_summary = historical_candidate_summary.sort_values(
        ["population_match", "historical_396_match", "oof_hint"],
        ascending=False,
    )

display(historical_candidate_summary.head(15))




,path,rows,response_unique,objective_column,objective_count,population_match,historical_396_match,oof_hint
2,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,35072,True,objective_key,396,True,True,True
8,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,35072,True,objective_key,396,True,True,True
16,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,35072,True,objective_key,396,True,True,False
37,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,35072,True,objective_key,396,True,True,False
109,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,35072,True,objective_key,396,True,True,False
121,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,35072,True,objective_key,396,True,True,False
1,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,35072,True,learning_objective,398,True,False,True
3,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,35072,True,objective_frequency,141,True,False,True
4,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,35072,True,objectives_per_session,9,True,False,True
5,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,35072,True,objective_words,13,True,False,True


In [23]:
historical_reference = None
objective_reconciliation_detail = pd.DataFrame()
historical_collapse_groups = pd.DataFrame()

if not historical_candidate_summary.empty:
    eligible = historical_candidate_summary[
        historical_candidate_summary["population_match"]
        & historical_candidate_summary["response_unique"]
    ]

    preferred = eligible[eligible["historical_396_match"]]
    selected = preferred.iloc[0] if not preferred.empty else (eligible.iloc[0] if not eligible.empty else None)

    if selected is not None:
        historical_path = Path(selected["path"])
        historical_column = selected["objective_column"]

        if historical_path.suffix.lower() == ".csv":
            historical_reference = pd.read_csv(
                historical_path,
                usecols=["response_id", historical_column],
            )
        else:
            historical_reference = pd.read_parquet(
                historical_path,
                columns=["response_id", historical_column],
            )

        historical_reference = historical_reference.rename(
            columns={historical_column: "historical_objective_key"}
        )

        objective_reconciliation_detail = objective_work[
            ["response_id", "objective_id_raw", "objective_raw", "objective_safe_norm", "objective_case_key"]
        ].merge(
            historical_reference,
            on="response_id",
            how="left",
            validate="one_to_one",
        )

        historical_collapse_groups = (
            objective_reconciliation_detail
            .groupby("historical_objective_key", dropna=False)
            .agg(
                current_objective_ids=("objective_id_raw", "nunique"),
                current_raw_texts=("objective_raw", "nunique"),
                response_count=("response_id", "size"),
            )
            .reset_index()
        )

        historical_collapse_groups = historical_collapse_groups[
            historical_collapse_groups["current_objective_ids"] > 1
        ]

display(historical_collapse_groups)

,historical_objective_key,current_objective_ids,current_raw_texts,response_count
15,086a53d8d71bfbdf,2,2,16
293,c04caa43183dccf3,2,2,136


In [24]:
current_id_count = objective_work["objective_id_raw"].nunique()
current_raw_count = objective_work["objective_raw"].nunique()
current_safe_count = objective_work["objective_safe_norm"].nunique()
current_case_count = objective_work["objective_case_key"].nunique()

current_population_checks = pd.DataFrame([
    check_row(
        "Objective ID → raw text is one-to-one",
        current_objective_catalogue["raw_text_variants"].eq(1).all(),
        int(current_objective_catalogue["raw_text_variants"].gt(1).sum())
    ),
    check_row(
        "Objective ID → safe text is deterministic",
        current_objective_catalogue["safe_text_variants"].eq(1).all(),
        int(current_objective_catalogue["safe_text_variants"].gt(1).sum())
    ),
    check_row(
        "Raw objective text → ID is one-to-one",
        raw_to_id_conflicts == 0,
        int(raw_to_id_conflicts)
    ),
    check_row(
        "All objectives have fold coverage",
        current_objective_catalogue["fold_count"].gt(0).all(),
        int(current_objective_catalogue["fold_count"].eq(0).sum())
    ),
    check_row(
        "Objective response accounting complete",
        current_objective_catalogue["response_count"].sum() == n_responses,
        f"{current_objective_catalogue['response_count'].sum()} / {n_responses}"
    ),
    check_row(
        "Safe-normalized objective identities unchanged",
        current_raw_count == current_safe_count,
        f"raw={current_raw_count}, safe={current_safe_count}",
        required=False
    ),
    check_row(
        "Safe text → ID remains one-to-one",
        safe_to_id_conflicts == 0,
        int(safe_to_id_conflicts),
        required=False
    ),
])

current_failures = current_population_checks[
    current_population_checks["required"] & ~current_population_checks["passed"]
]

CURRENT_OBJECTIVE_POPULATION_VALID = current_failures.empty

if historical_reference is None:
    HISTORICAL_OBJECTIVE_RECONCILIATION = "REFERENCE_UNAVAILABLE"
    historical_objective_count = None
    collapse_reduction = None

else:
    historical_objective_count = historical_reference["historical_objective_key"].nunique(dropna=True)

    current_to_historical = (
        objective_reconciliation_detail
        .groupby("objective_id_raw")["historical_objective_key"]
        .nunique(dropna=False)
    )

    collapse_reduction = (
        historical_collapse_groups["current_objective_ids"] - 1
    ).sum() if not historical_collapse_groups.empty else 0

    historical_coverage_ok = objective_reconciliation_detail["historical_objective_key"].notna().all()
    mapping_stable = current_to_historical.eq(1).all()

    if (
        historical_objective_count == 396
        and historical_coverage_ok
        and mapping_stable
        and collapse_reduction == current_id_count - historical_objective_count
    ):
        HISTORICAL_OBJECTIVE_RECONCILIATION = "EXPLAINED_AS_HISTORICAL_COLLAPSE"
    else:
        HISTORICAL_OBJECTIVE_RECONCILIATION = "UNRESOLVED"

OBJECTIVE_RECONCILIATION_READY = CURRENT_OBJECTIVE_POPULATION_VALID

reconciliation_summary = pd.DataFrame({
    "item": [
        "Current objective IDs",
        "Current raw texts",
        "Current safe texts",
        "Current case keys",
        "Historical objective count",
        "Historical collapse groups",
        "Historical collapse reduction",
        "Current population valid",
        "Historical reconciliation",
        "Required failures",
        "OBJECTIVE_RECONCILIATION_READY",
    ],
    "value": [
        current_id_count,
        current_raw_count,
        current_safe_count,
        current_case_count,
        historical_objective_count,
        len(historical_collapse_groups),
        collapse_reduction,
        CURRENT_OBJECTIVE_POPULATION_VALID,
        HISTORICAL_OBJECTIVE_RECONCILIATION,
        len(current_failures),
        OBJECTIVE_RECONCILIATION_READY,
    ],
})

display(current_population_checks)
display(reconciliation_summary)

assert OBJECTIVE_RECONCILIATION_READY, (
    "Section 1.6 failed.\n\n"
    + current_failures[["check", "detail"]].to_string(index=False)
)

,check,required,passed,detail
0,Objective ID → raw text is one-to-one,True,True,0
1,Objective ID → safe text is deterministic,True,True,0
2,Raw objective text → ID is one-to-one,True,True,0
3,All objectives have fold coverage,True,True,0
4,Objective response accounting complete,True,True,35072 / 35072
5,Safe-normalized objective identities unchanged,False,True,"raw=398, safe=398"
6,Safe text → ID remains one-to-one,False,True,0


,item,value
0,Current objective IDs,398
1,Current raw texts,398
2,Current safe texts,398
3,Current case keys,396
4,Historical objective count,396
5,Historical collapse groups,2
6,Historical collapse reduction,2
7,Current population valid,True
8,Historical reconciliation,EXPLAINED_AS_HISTORICAL_COLLAPSE
9,Required failures,0


# Section 1.7 — Transcript File Inventory

This section inventories the internal structure of every registered transcript CSV file.
Each file is checked for readability, row count, column schema, and contained session IDs.
The required transcript fields are `session_id`, `utterance_id`, `role`, `content`, and `timestamp`.
Extra columns are allowed, but missing required columns are explicitly flagged.
Session-to-file structure and filename-to-session consistency are recorded without correction.
The Section 1.2 fingerprint manifest is reused to detect source drift and exact duplicate files.
No role analysis, timestamp parsing, utterance ordering, or text normalization is performed here.
Individual file problems are carried forward for coverage analysis in Section 1.8.
No transcript rows are modified or deleted.
The section ends with the `TRANSCRIPT_INVENTORY_READY` gate.

In [25]:
import csv

assert OBJECTIVE_RECONCILIATION_READY, "Section 1.6 must pass before Section 1.7."

REQUIRED_TRANSCRIPT_COLUMNS = {
    "session_id",
    "utterance_id",
    "role",
    "content",
    "timestamp",
}

def inspect_transcript_file(path, root):
    relative_path = path.relative_to(root).as_posix()
    result = {
        "relative_path": relative_path,
        "file_name": path.name,
        "file_stem": path.stem,
        "size_bytes": path.stat().st_size,
        "modified_time": datetime.fromtimestamp(path.stat().st_mtime).astimezone().isoformat(),
        "row_count": None,
        "column_count": None,
        "columns_text": None,
        "schema_signature": None,
        "session_count_inside": None,
        "session_ids": None,
        "malformed_row_count": None,
        "read_status": "READ_ERROR",
        "schema_status": "UNKNOWN",
        "session_structure": "UNKNOWN",
        "filename_session_match": pd.NA,
        "error": None,
    }

    try:
        with open(path, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.reader(f, strict=True)
            header = next(reader, None)

            if not header or not any(header):
                result["read_status"] = "INVALID_HEADER"
                result["schema_status"] = "INVALID_HEADER"
                return result

            duplicate_columns = len(header) != len(set(header))
            missing_columns = REQUIRED_TRANSCRIPT_COLUMNS - set(header)
            extra_columns = set(header) - REQUIRED_TRANSCRIPT_COLUMNS

            result["column_count"] = len(header)
            result["columns_text"] = " | ".join(header)
            result["schema_signature"] = hashlib.sha256(
                "\x1f".join(header).encode("utf-8")
            ).hexdigest()

            session_index = header.index("session_id") if "session_id" in header else None
            session_ids = set()
            row_count = 0
            malformed_rows = 0

            for row in reader:
                if not row or all(cell == "" for cell in row):
                    continue

                row_count += 1

                if len(row) != len(header):
                    malformed_rows += 1

                if session_index is not None and session_index < len(row):
                    session_value = row[session_index]

                    if session_value != "":
                        session_ids.add(session_value)

            result["row_count"] = row_count
            result["session_count_inside"] = len(session_ids)
            result["session_ids"] = tuple(sorted(session_ids))
            result["malformed_row_count"] = malformed_rows
            result["read_status"] = "EMPTY_FILE" if row_count == 0 else "READABLE"

            if duplicate_columns:
                result["schema_status"] = "DUPLICATE_COLUMN_NAMES"
            elif missing_columns:
                result["schema_status"] = "MISSING_REQUIRED_COLUMNS"
            elif malformed_rows:
                result["schema_status"] = "ROW_WIDTH_MISMATCH"
            elif extra_columns:
                result["schema_status"] = "VALID_WITH_EXTRA_COLUMNS"
            else:
                result["schema_status"] = "VALID"

            if len(session_ids) == 0:
                result["session_structure"] = "NO_SESSION"
            elif len(session_ids) == 1:
                result["session_structure"] = "SINGLE_SESSION"
                result["filename_session_match"] = path.stem == next(iter(session_ids))
            else:
                result["session_structure"] = "MULTI_SESSION_FILE"

    except Exception as e:
        result["error"] = f"{type(e).__name__}: {e}"

    return result


current_transcript_files = sorted(
    TRANSCRIPT_ROOT.rglob("*.csv"),
    key=lambda p: p.relative_to(TRANSCRIPT_ROOT).as_posix(),
)

transcript_file_inventory = pd.DataFrame([
    inspect_transcript_file(path, TRANSCRIPT_ROOT)
    for path in current_transcript_files
])

display(
    transcript_file_inventory[
        [
            "relative_path",
            "row_count",
            "column_count",
            "session_count_inside",
            "read_status",
            "schema_status",
            "session_structure",
            "filename_session_match",
        ]
    ].head(10)
)

,relative_path,row_count,column_count,session_count_inside,read_status,schema_status,session_structure,filename_session_match
0,aaaedit.csv,254,5,1,READABLE,VALID,SINGLE_SESSION,True
1,aaaptjd.csv,360,5,1,READABLE,VALID,SINGLE_SESSION,True
2,aabkeov.csv,281,5,1,READABLE,VALID,SINGLE_SESSION,True
3,aacggvb.csv,235,5,1,READABLE,VALID,SINGLE_SESSION,True
4,aadexbc.csv,104,5,1,READABLE,VALID,SINGLE_SESSION,True
5,aadinwu.csv,233,5,1,READABLE,VALID,SINGLE_SESSION,True
6,aadljmq.csv,372,5,1,READABLE,VALID,SINGLE_SESSION,True
7,aadmino.csv,195,5,1,READABLE,VALID,SINGLE_SESSION,True
8,aadsgow.csv,265,5,1,READABLE,VALID,SINGLE_SESSION,True
9,aadylxv.csv,235,5,1,READABLE,VALID,SINGLE_SESSION,True


In [26]:
schema_distribution = (
    transcript_file_inventory
    .groupby(
        ["schema_signature", "columns_text", "schema_status"],
        dropna=False
    )
    .size()
    .reset_index(name="file_count")
    .sort_values("file_count", ascending=False)
)

session_source_rows = []

for row in transcript_file_inventory.itertuples(index=False):
    if not row.session_ids:
        continue

    for session_id in row.session_ids:
        session_source_rows.append({
            "session_id": session_id,
            "relative_file": row.relative_path,
            "schema_status": row.schema_status,
            "read_status": row.read_status,
        })

transcript_session_source_map = pd.DataFrame(session_source_rows)

if transcript_session_source_map.empty:
    session_source_summary = pd.DataFrame(
        columns=["session_id", "source_file_count", "source_status"]
    )
else:
    session_source_summary = (
        transcript_session_source_map
        .groupby("session_id", as_index=False)
        .agg(source_file_count=("relative_file", "nunique"))
    )

    session_source_summary["source_status"] = session_source_summary[
        "source_file_count"
    ].map(lambda n: "SINGLE_SOURCE" if n == 1 else "MULTI_SOURCE_REVIEW")


hash_manifest = transcript_fingerprint_manifest[
    ["relative_path", "size_bytes", "modified_time", "sha256"]
].rename(columns={
    "size_bytes": "fingerprint_size_bytes",
    "modified_time": "fingerprint_modified_time",
})

drift_audit = transcript_file_inventory[
    ["relative_path", "size_bytes", "modified_time"]
].merge(
    hash_manifest,
    on="relative_path",
    how="outer",
    indicator=True,
    validate="one_to_one",
)

drift_audit["size_changed"] = (
    drift_audit["size_bytes"].notna()
    & drift_audit["fingerprint_size_bytes"].notna()
    & drift_audit["size_bytes"].ne(drift_audit["fingerprint_size_bytes"])
)

drift_audit["mtime_changed"] = (
    drift_audit["modified_time"].notna()
    & drift_audit["fingerprint_modified_time"].notna()
    & drift_audit["modified_time"].ne(drift_audit["fingerprint_modified_time"])
)

hash_groups = (
    transcript_fingerprint_manifest
    .dropna(subset=["sha256"])
    .groupby("sha256")
    .agg(
        file_count=("relative_path", "size"),
        files=("relative_path", lambda x: tuple(sorted(x))),
    )
    .reset_index()
)

duplicate_file_hash_groups = hash_groups[
    hash_groups["file_count"] > 1
].copy()

single_session_files = transcript_file_inventory[
    transcript_file_inventory["session_structure"] == "SINGLE_SESSION"
]

filename_match_rate = (
    single_session_files["filename_session_match"].mean()
    if len(single_session_files)
    else None
)

inventory_summary = pd.DataFrame({
    "item": [
        "Transcript files scanned",
        "Readable files",
        "Empty files",
        "Read errors",
        "Valid schema files",
        "Valid-with-extra schema files",
        "Missing-schema files",
        "Row-width mismatch files",
        "Schema variants",
        "Unique internal sessions",
        "Single-source sessions",
        "Multi-source sessions",
        "Multi-session files",
        "Exact duplicate file-hash groups",
        "Filename-session match rate",
        "Files added since Section 1.2",
        "Files removed since Section 1.2",
        "Files with size drift",
        "Files with mtime drift",
    ],
    "value": [
        len(transcript_file_inventory),
        int((transcript_file_inventory["read_status"] == "READABLE").sum()),
        int((transcript_file_inventory["read_status"] == "EMPTY_FILE").sum()),
        int((transcript_file_inventory["read_status"] == "READ_ERROR").sum()),
        int((transcript_file_inventory["schema_status"] == "VALID").sum()),
        int((transcript_file_inventory["schema_status"] == "VALID_WITH_EXTRA_COLUMNS").sum()),
        int((transcript_file_inventory["schema_status"] == "MISSING_REQUIRED_COLUMNS").sum()),
        int((transcript_file_inventory["schema_status"] == "ROW_WIDTH_MISMATCH").sum()),
        transcript_file_inventory["schema_signature"].nunique(dropna=True),
        transcript_session_source_map["session_id"].nunique() if not transcript_session_source_map.empty else 0,
        int((session_source_summary["source_file_count"] == 1).sum()) if not session_source_summary.empty else 0,
        int((session_source_summary["source_file_count"] > 1).sum()) if not session_source_summary.empty else 0,
        int((transcript_file_inventory["session_structure"] == "MULTI_SESSION_FILE").sum()),
        len(duplicate_file_hash_groups),
        round(float(filename_match_rate), 6) if filename_match_rate is not None else None,
        int((drift_audit["_merge"] == "left_only").sum()),
        int((drift_audit["_merge"] == "right_only").sum()),
        int(drift_audit["size_changed"].sum()),
        int(drift_audit["mtime_changed"].sum()),
    ],
})

display(schema_distribution)
display(inventory_summary)

,schema_signature,columns_text,schema_status,file_count
0,5cccec9df92a667e146d9a7178caa50e74b30bfbe4dfad...,session_id | utterance_id | role | content | t...,VALID,22821


,item,value
0,Transcript files scanned,22821.0
1,Readable files,22821.0
2,Empty files,0.0
3,Read errors,0.0
4,Valid schema files,22821.0
5,Valid-with-extra schema files,0.0
6,Missing-schema files,0.0
7,Row-width mismatch files,0.0
8,Schema variants,1.0
9,Unique internal sessions,22821.0


In [27]:
files_scanned = len(transcript_file_inventory)
fingerprinted_files = len(transcript_fingerprint_manifest)

read_errors = int((transcript_file_inventory["read_status"] == "READ_ERROR").sum())
empty_files = int((transcript_file_inventory["read_status"] == "EMPTY_FILE").sum())

valid_schema_mask = transcript_file_inventory["schema_status"].isin(
    ["VALID", "VALID_WITH_EXTRA_COLUMNS"]
)

valid_schema_files = int(valid_schema_mask.sum())
missing_schema_files = int(
    (transcript_file_inventory["schema_status"] == "MISSING_REQUIRED_COLUMNS").sum()
)

multi_session_files = int(
    (transcript_file_inventory["session_structure"] == "MULTI_SESSION_FILE").sum()
)

no_session_files = int(
    (transcript_file_inventory["session_structure"] == "NO_SESSION").sum()
)

files_added = int((drift_audit["_merge"] == "left_only").sum())
files_removed = int((drift_audit["_merge"] == "right_only").sum())
size_drift_files = int(drift_audit["size_changed"].sum())

all_files_classified = (
    transcript_file_inventory["read_status"].notna().all()
    and transcript_file_inventory["schema_status"].notna().all()
    and transcript_file_inventory["session_structure"].notna().all()
)

transcript_inventory_checks = pd.DataFrame([
    check_row(
        "All fingerprinted transcript files scanned",
        files_scanned == fingerprinted_files,
        f"scanned={files_scanned}, fingerprinted={fingerprinted_files}"
    ),
    check_row(
        "No transcript files added after fingerprinting",
        files_added == 0,
        files_added
    ),
    check_row(
        "No transcript files removed after fingerprinting",
        files_removed == 0,
        files_removed
    ),
    check_row(
        "No transcript file-size drift",
        size_drift_files == 0,
        size_drift_files
    ),
    check_row(
        "All transcript files structurally classified",
        all_files_classified,
        f"classified={all_files_classified}"
    ),
    check_row(
        "Required transcript schema is established",
        valid_schema_files > 0,
        f"valid_schema_files={valid_schema_files}"
    ),
    check_row(
        "All transcript files readable",
        read_errors == 0,
        read_errors,
        required=False
    ),
    check_row(
        "No empty transcript files",
        empty_files == 0,
        empty_files,
        required=False
    ),
    check_row(
        "No missing-schema transcript files",
        missing_schema_files == 0,
        missing_schema_files,
        required=False
    ),
    check_row(
        "No transcript files without session IDs",
        no_session_files == 0,
        no_session_files,
        required=False
    ),
    check_row(
        "No multi-session transcript files",
        multi_session_files == 0,
        multi_session_files,
        required=False
    ),
    check_row(
        "No multi-source sessions",
        int((session_source_summary["source_file_count"] > 1).sum()) == 0
        if not session_source_summary.empty else False,
        int((session_source_summary["source_file_count"] > 1).sum())
        if not session_source_summary.empty else "no session mapping",
        required=False
    ),
])

inventory_failures = transcript_inventory_checks[
    transcript_inventory_checks["required"] & ~transcript_inventory_checks["passed"]
]

inventory_warnings = transcript_inventory_checks[
    ~transcript_inventory_checks["required"] & ~transcript_inventory_checks["passed"]
]

TRANSCRIPT_INVENTORY_READY = inventory_failures.empty

transcript_gate_summary = pd.DataFrame({
    "item": [
        "Transcript files scanned",
        "Unique internal sessions",
        "Readable files",
        "Valid schema files",
        "Read errors",
        "Empty files",
        "Missing-schema files",
        "Multi-session files",
        "Multi-source sessions",
        "Exact duplicate hash groups",
        "Required failures",
        "Non-blocking warnings",
        "TRANSCRIPT_INVENTORY_READY",
    ],
    "value": [
        files_scanned,
        transcript_session_source_map["session_id"].nunique()
        if not transcript_session_source_map.empty else 0,
        int((transcript_file_inventory["read_status"] == "READABLE").sum()),
        valid_schema_files,
        read_errors,
        empty_files,
        missing_schema_files,
        multi_session_files,
        int((session_source_summary["source_file_count"] > 1).sum())
        if not session_source_summary.empty else 0,
        len(duplicate_file_hash_groups),
        len(inventory_failures),
        len(inventory_warnings),
        TRANSCRIPT_INVENTORY_READY,
    ],
})

display(transcript_inventory_checks)
display(transcript_gate_summary)

assert TRANSCRIPT_INVENTORY_READY, (
    "Section 1.7 failed.\n\n"
    + inventory_failures[["check", "detail"]].to_string(index=False)
)

,check,required,passed,detail
0,All fingerprinted transcript files scanned,True,True,"scanned=22821, fingerprinted=22821"
1,No transcript files added after fingerprinting,True,True,0
2,No transcript files removed after fingerprinting,True,True,0
3,No transcript file-size drift,True,True,0
4,All transcript files structurally classified,True,True,classified=True
5,Required transcript schema is established,True,True,valid_schema_files=22821
6,All transcript files readable,False,True,0
7,No empty transcript files,False,True,0
8,No missing-schema transcript files,False,True,0
9,No transcript files without session IDs,False,True,0


,item,value
0,Transcript files scanned,22821
1,Unique internal sessions,22821
2,Readable files,22821
3,Valid schema files,22821
4,Read errors,0
5,Empty files,0
6,Missing-schema files,0
7,Multi-session files,0
8,Multi-source sessions,0
9,Exact duplicate hash groups,0


# Section 1.8 — Transcript Session Coverage

This section verifies that every labelled response session has usable transcript evidence.
Response sessions are compared directly with the internal session IDs found in transcript files.
A transcript source is usable only when it is readable, non-empty, and has the required schema.
Coverage is classified as matched, missing, unusable, orphan, or ambiguous multi-source.
Internal `session_id` is the coverage authority; transcript filenames are provenance only.
The session-level coverage table is joined back to all response rows using a many-to-one contract.
No response row may be lost or multiplied during this linkage.
Transcript utterances are not loaded or duplicated into the response table.
Orphan transcript sessions are documented but do not automatically block training coverage.
The section ends with the `TRANSCRIPT_COVERAGE_READY` gate.

In [29]:
assert TRANSCRIPT_INVENTORY_READY, "Section 1.7 must pass before Section 1.8."

VALID_TRANSCRIPT_SCHEMAS = {"VALID", "VALID_WITH_EXTRA_COLUMNS"}

response_session_population = (
    responses_base.groupby("session_id", as_index=False)
    .agg(
        response_count=("response_id", "size"),
        objective_count=("objective_id_raw", "nunique"),
    )
)

required_map_columns = {"session_id", "relative_file", "read_status", "schema_status"}
assert required_map_columns.issubset(transcript_session_source_map.columns), (
    f"Missing columns in transcript_session_source_map: "
    f"{required_map_columns - set(transcript_session_source_map.columns)}"
)

transcript_source_detail = transcript_session_source_map.merge(
    transcript_file_inventory[
        ["relative_path", "row_count"]
    ],
    left_on="relative_file",
    right_on="relative_path",
    how="left",
    validate="many_to_one",
)

transcript_source_detail["usable_source"] = (
    transcript_source_detail["read_status"].eq("READABLE")
    & transcript_source_detail["schema_status"].isin(VALID_TRANSCRIPT_SCHEMAS)
    & transcript_source_detail["row_count"].fillna(0).gt(0)
)

transcript_session_population = (
    transcript_source_detail.groupby("session_id", as_index=False)
    .agg(
        source_file_count=("relative_file", "nunique"),
        usable_source_count=("usable_source", "sum"),
        source_files=("relative_file", lambda x: tuple(sorted(set(x)))),
    )
)

session_coverage = response_session_population.merge(
    transcript_session_population,
    on="session_id",
    how="outer",
    indicator=True,
    validate="one_to_one",
)

session_coverage["response_exists"] = session_coverage["_merge"].ne("right_only")
session_coverage["transcript_exists"] = session_coverage["_merge"].ne("left_only")

for column in ["response_count", "objective_count", "source_file_count", "usable_source_count"]:
    session_coverage[column] = session_coverage[column].fillna(0).astype(int)

def coverage_status(row):
    if not row["response_exists"]:
        return "ORPHAN_TRANSCRIPT"

    if not row["transcript_exists"]:
        return "MISSING_TRANSCRIPT"

    if row["source_file_count"] > 1:
        return "MULTI_SOURCE_AMBIGUOUS"

    if row["usable_source_count"] == 0:
        return "UNUSABLE_TRANSCRIPT"

    return "MATCHED"

session_coverage["coverage_status"] = session_coverage.apply(coverage_status, axis=1)
session_coverage = session_coverage.drop(columns="_merge")

coverage_status_counts = (
    session_coverage["coverage_status"]
    .value_counts()
    .rename_axis("coverage_status")
    .reset_index(name="session_count")
)

display(coverage_status_counts)
display(session_coverage.head())

,coverage_status,session_count
0,MATCHED,22821


,session_id,response_count,objective_count,source_file_count,usable_source_count,source_files,response_exists,transcript_exists,coverage_status
0,aaaedit,2,2,1,1,"(aaaedit.csv,)",True,True,MATCHED
1,aaaptjd,1,1,1,1,"(aaaptjd.csv,)",True,True,MATCHED
2,aabkeov,1,1,1,1,"(aabkeov.csv,)",True,True,MATCHED
3,aacggvb,1,1,1,1,"(aacggvb.csv,)",True,True,MATCHED
4,aadexbc,2,2,1,1,"(aadexbc.csv,)",True,True,MATCHED


In [30]:
coverage_columns = [
    "session_id",
    "source_file_count",
    "usable_source_count",
    "coverage_status",
]

response_transcript_link = responses_base[
    ["response_id", "session_id"]
].merge(
    session_coverage[coverage_columns],
    on="session_id",
    how="left",
    validate="many_to_one",
)

response_rows_before = len(responses_base)
response_rows_after = len(response_transcript_link)

matched_response_rows = int(
    response_transcript_link["coverage_status"].eq("MATCHED").sum()
)

uncovered_response_rows = int(
    response_transcript_link["coverage_status"].ne("MATCHED").sum()
)

missing_coverage_rows = int(
    response_transcript_link["coverage_status"].isna().sum()
)

response_link_summary = pd.DataFrame({
    "item": [
        "Response rows before coverage join",
        "Response rows after coverage join",
        "Unique response IDs after join",
        "Matched response rows",
        "Uncovered response rows",
        "Missing coverage classifications",
    ],
    "value": [
        response_rows_before,
        response_rows_after,
        response_transcript_link["response_id"].nunique(),
        matched_response_rows,
        uncovered_response_rows,
        missing_coverage_rows,
    ],
})

session_coverage_summary = pd.DataFrame({
    "item": [
        "Response sessions",
        "Transcript sessions",
        "Matched sessions",
        "Missing transcript sessions",
        "Unusable transcript sessions",
        "Ambiguous multi-source sessions",
        "Orphan transcript sessions",
    ],
    "value": [
        len(response_session_population),
        len(transcript_session_population),
        int(session_coverage["coverage_status"].eq("MATCHED").sum()),
        int(session_coverage["coverage_status"].eq("MISSING_TRANSCRIPT").sum()),
        int(session_coverage["coverage_status"].eq("UNUSABLE_TRANSCRIPT").sum()),
        int(session_coverage["coverage_status"].eq("MULTI_SOURCE_AMBIGUOUS").sum()),
        int(session_coverage["coverage_status"].eq("ORPHAN_TRANSCRIPT").sum()),
    ],
})

display(session_coverage_summary)
display(response_link_summary)

,item,value
0,Response sessions,22821
1,Transcript sessions,22821
2,Matched sessions,22821
3,Missing transcript sessions,0
4,Unusable transcript sessions,0
5,Ambiguous multi-source sessions,0
6,Orphan transcript sessions,0


,item,value
0,Response rows before coverage join,35072
1,Response rows after coverage join,35072
2,Unique response IDs after join,35072
3,Matched response rows,35072
4,Uncovered response rows,0
5,Missing coverage classifications,0


In [31]:
missing_sessions = int(
    session_coverage["coverage_status"].eq("MISSING_TRANSCRIPT").sum()
)

unusable_sessions = int(
    session_coverage["coverage_status"].eq("UNUSABLE_TRANSCRIPT").sum()
)

ambiguous_sessions = int(
    session_coverage["coverage_status"].eq("MULTI_SOURCE_AMBIGUOUS").sum()
)

orphan_sessions = int(
    session_coverage["coverage_status"].eq("ORPHAN_TRANSCRIPT").sum()
)

filename_mismatches = int(
    (
        transcript_file_inventory["session_structure"].eq("SINGLE_SESSION")
        & transcript_file_inventory["filename_session_match"].eq(False)
    ).sum()
)

coverage_checks = pd.DataFrame([
    check_row(
        "Every response session classified",
        session_coverage.loc[
            session_coverage["response_exists"], "coverage_status"
        ].notna().all(),
        len(response_session_population)
    ),
    check_row(
        "No labelled session missing transcript",
        missing_sessions == 0,
        missing_sessions
    ),
    check_row(
        "No labelled session has unusable transcript",
        unusable_sessions == 0,
        unusable_sessions
    ),
    check_row(
        "No ambiguous multi-source labelled session",
        ambiguous_sessions == 0,
        ambiguous_sessions
    ),
    check_row(
        "Response row count preserved",
        response_rows_before == response_rows_after,
        f"{response_rows_before} / {response_rows_after}"
    ),
    check_row(
        "Response IDs remain unique after coverage join",
        response_transcript_link["response_id"].is_unique,
        response_transcript_link["response_id"].duplicated().sum()
    ),
    check_row(
        "Every response has usable transcript coverage",
        uncovered_response_rows == 0 and missing_coverage_rows == 0,
        f"uncovered={uncovered_response_rows}, missing={missing_coverage_rows}"
    ),
    check_row(
        "No orphan transcript sessions",
        orphan_sessions == 0,
        orphan_sessions,
        required=False
    ),
    check_row(
        "Transcript filenames match internal session IDs",
        filename_mismatches == 0,
        filename_mismatches,
        required=False
    ),
])

coverage_failures = coverage_checks[
    coverage_checks["required"] & ~coverage_checks["passed"]
]

coverage_warnings = coverage_checks[
    ~coverage_checks["required"] & ~coverage_checks["passed"]
]

TRANSCRIPT_COVERAGE_READY = coverage_failures.empty

coverage_gate_summary = pd.DataFrame({
    "item": [
        "Response sessions",
        "Transcript sessions",
        "Matched sessions",
        "Missing transcript sessions",
        "Unusable transcript sessions",
        "Ambiguous multi-source sessions",
        "Orphan transcript sessions",
        "Response rows",
        "Matched response rows",
        "Uncovered response rows",
        "Required failures",
        "Non-blocking warnings",
        "TRANSCRIPT_COVERAGE_READY",
    ],
    "value": [
        len(response_session_population),
        len(transcript_session_population),
        int(session_coverage["coverage_status"].eq("MATCHED").sum()),
        missing_sessions,
        unusable_sessions,
        ambiguous_sessions,
        orphan_sessions,
        response_rows_before,
        matched_response_rows,
        uncovered_response_rows,
        len(coverage_failures),
        len(coverage_warnings),
        TRANSCRIPT_COVERAGE_READY,
    ],
})

display(coverage_checks)
display(coverage_gate_summary)

assert TRANSCRIPT_COVERAGE_READY, (
    "Section 1.8 failed.\n\n"
    + coverage_failures[["check", "detail"]].to_string(index=False)
)

,check,required,passed,detail
0,Every response session classified,True,True,22821
1,No labelled session missing transcript,True,True,0
2,No labelled session has unusable transcript,True,True,0
3,No ambiguous multi-source labelled session,True,True,0
4,Response row count preserved,True,True,35072 / 35072
5,Response IDs remain unique after coverage join,True,True,0
6,Every response has usable transcript coverage,True,True,"uncovered=0, missing=0"
7,No orphan transcript sessions,False,True,0
8,Transcript filenames match internal session IDs,False,True,0


,item,value
0,Response sessions,22821
1,Transcript sessions,22821
2,Matched sessions,22821
3,Missing transcript sessions,0
4,Unusable transcript sessions,0
5,Ambiguous multi-source sessions,0
6,Orphan transcript sessions,0
7,Response rows,35072
8,Matched response rows,35072
9,Uncovered response rows,0


# Section 1.9 — Frozen Fold Contract

This section validates the fixed grouped-validation manifest used by the entire project.
Every official response must appear exactly once in the frozen fold assignment.
The response-to-session relationship must agree with the authoritative response foundation.
Every tutoring session must remain entirely inside one fold to prevent session leakage.
The expected fold domain is fixed to five folds: `0, 1, 2, 3, 4`.
Assignments in `responses_base` are compared exactly with the original frozen manifest.
Fold-level response, session, and target distributions are recorded only as diagnostics.
The frozen manifest is never regenerated or rebalanced in this notebook.
Its established fingerprint is retained for downstream reproducibility.
The section ends with the `FROZEN_FOLD_CONTRACT_READY` gate.

In [32]:
assert TRANSCRIPT_COVERAGE_READY, "Section 1.8 must pass before Section 1.9."

expected_response_ids = set(responses_base["response_id"])
manifest_response_ids = set(frozen_folds["response_id"])

missing_fold_responses = expected_response_ids - manifest_response_ids
extra_fold_responses = manifest_response_ids - expected_response_ids

duplicate_response_rows = int(
    frozen_folds["response_id"].duplicated(keep=False).sum()
)

fold_response_map = (
    frozen_folds.groupby("response_id", as_index=False)
    .agg(
        manifest_session_id=("session_id", "first"),
        session_count=("session_id", "nunique"),
        fold_count=("fold", "nunique"),
    )
)

response_session_audit = responses_base[
    ["response_id", "session_id"]
].merge(
    fold_response_map,
    on="response_id",
    how="left",
    validate="one_to_one",
)

response_session_mismatches = int(
    (
        response_session_audit["manifest_session_id"].notna()
        & response_session_audit["session_id"].ne(
            response_session_audit["manifest_session_id"]
        )
    ).sum()
)

responses_with_multiple_sessions = int(
    fold_response_map["session_count"].gt(1).sum()
)

responses_with_multiple_folds = int(
    fold_response_map["fold_count"].gt(1).sum()
)

session_fold_audit = (
    frozen_folds.groupby("session_id", as_index=False)
    .agg(
        response_count=("response_id", "size"),
        fold_count=("fold", "nunique"),
        fold=("fold", "first"),
    )
)

cross_fold_sessions = session_fold_audit[
    session_fold_audit["fold_count"] > 1
].copy()

assignment_summary = pd.DataFrame({
    "item": [
        "Foundation responses",
        "Manifest responses",
        "Missing response assignments",
        "Extra response assignments",
        "Duplicate response rows",
        "Response-session mismatches",
        "Responses mapped to multiple sessions",
        "Responses mapped to multiple folds",
        "Manifest sessions",
        "Cross-fold sessions",
        "Maximum folds per session",
    ],
    "value": [
        len(responses_base),
        len(frozen_folds),
        len(missing_fold_responses),
        len(extra_fold_responses),
        duplicate_response_rows,
        response_session_mismatches,
        responses_with_multiple_sessions,
        responses_with_multiple_folds,
        len(session_fold_audit),
        len(cross_fold_sessions),
        int(session_fold_audit["fold_count"].max()),
    ],
})

display(assignment_summary)

,item,value
0,Foundation responses,35072
1,Manifest responses,35072
2,Missing response assignments,0
3,Extra response assignments,0
4,Duplicate response rows,0
5,Response-session mismatches,0
6,Responses mapped to multiple sessions,0
7,Responses mapped to multiple folds,0
8,Manifest sessions,22821
9,Cross-fold sessions,0


In [33]:
EXPECTED_FOLDS = {0, 1, 2, 3, 4}

observed_folds = set(
    frozen_folds["fold"].dropna().astype(int).unique()
)

missing_fold_ids = EXPECTED_FOLDS - observed_folds
unexpected_fold_ids = observed_folds - EXPECTED_FOLDS
null_fold_rows = int(frozen_folds["fold"].isna().sum())

foundation_assignments = set(
    responses_base[
        ["response_id", "session_id", "fold"]
    ].itertuples(index=False, name=None)
)

manifest_assignments = set(
    frozen_folds[
        ["response_id", "session_id", "fold"]
    ].itertuples(index=False, name=None)
)

foundation_only_assignments = (
    foundation_assignments - manifest_assignments
)

manifest_only_assignments = (
    manifest_assignments - foundation_assignments
)

fold_distribution = (
    responses_base.groupby("fold", as_index=False)
    .agg(
        response_count=("response_id", "size"),
        session_count=("session_id", "nunique"),
        positive_count=("target", lambda x: int((x == 1).sum())),
        negative_count=("target", lambda x: int((x == 0).sum())),
        positive_rate=("target", "mean"),
    )
    .sort_values("fold")
)

fold_distribution["positive_rate"] = (
    fold_distribution["positive_rate"].round(6)
)

FROZEN_FOLD_SHA256 = current_source_fingerprints[
    "frozen_fold_manifest_sha256"
]

fold_domain_summary = pd.DataFrame({
    "item": [
        "Expected fold count",
        "Observed fold count",
        "Observed fold IDs",
        "Missing fold IDs",
        "Unexpected fold IDs",
        "Null fold rows",
        "Foundation-only assignments",
        "Manifest-only assignments",
        "Frozen fold SHA256",
    ],
    "value": [
        len(EXPECTED_FOLDS),
        len(observed_folds),
        ",".join(map(str, sorted(observed_folds))),
        len(missing_fold_ids),
        len(unexpected_fold_ids),
        null_fold_rows,
        len(foundation_only_assignments),
        len(manifest_only_assignments),
        FROZEN_FOLD_SHA256,
    ],
})

display(fold_distribution)
display(fold_domain_summary)

,fold,response_count,session_count,positive_count,negative_count,positive_rate
0,0,6980,4590,4890,2090,0.700573
1,1,7018,4569,4905,2113,0.698917
2,2,7019,4570,4989,2030,0.710785
3,3,7011,4541,4905,2106,0.699615
4,4,7044,4551,4948,2096,0.702442


,item,value
0,Expected fold count,5
1,Observed fold count,5
2,Observed fold IDs,"0,1,2,3,4"
3,Missing fold IDs,0
4,Unexpected fold IDs,0
5,Null fold rows,0
6,Foundation-only assignments,0
7,Manifest-only assignments,0
8,Frozen fold SHA256,567fc95ebc781126c879e9e85bf91d38a697022889bd35...


In [34]:
fold_checks = pd.DataFrame([
    check_row(
        "Every foundation response has a fold assignment",
        len(missing_fold_responses) == 0,
        len(missing_fold_responses)
    ),
    check_row(
        "No extra response exists in fold manifest",
        len(extra_fold_responses) == 0,
        len(extra_fold_responses)
    ),
    check_row(
        "No duplicate response assignment",
        duplicate_response_rows == 0,
        duplicate_response_rows
    ),
    check_row(
        "Response-session mapping is consistent",
        response_session_mismatches == 0,
        response_session_mismatches
    ),
    check_row(
        "Each response maps to one session",
        responses_with_multiple_sessions == 0,
        responses_with_multiple_sessions
    ),
    check_row(
        "Each response maps to one fold",
        responses_with_multiple_folds == 0,
        responses_with_multiple_folds
    ),
    check_row(
        "No session crosses folds",
        len(cross_fold_sessions) == 0,
        len(cross_fold_sessions)
    ),
    check_row(
        "Maximum one fold per session",
        session_fold_audit["fold_count"].max() == 1,
        int(session_fold_audit["fold_count"].max())
    ),
    check_row(
        "Fold domain is exactly 0–4",
        observed_folds == EXPECTED_FOLDS,
        f"observed={sorted(observed_folds)}"
    ),
    check_row(
        "No null fold values",
        null_fold_rows == 0,
        null_fold_rows
    ),
    check_row(
        "Foundation assignments match frozen manifest",
        len(foundation_only_assignments) == 0
        and len(manifest_only_assignments) == 0,
        (
            f"foundation_only={len(foundation_only_assignments)}, "
            f"manifest_only={len(manifest_only_assignments)}"
        )
    ),
    check_row(
        "Frozen fold fingerprint available",
        bool(FROZEN_FOLD_SHA256),
        FROZEN_FOLD_SHA256
    ),
])

fold_failures = fold_checks[
    fold_checks["required"] & ~fold_checks["passed"]
]

FROZEN_FOLD_CONTRACT_READY = fold_failures.empty

fold_gate_summary = pd.DataFrame({
    "item": [
        "Foundation responses",
        "Manifest responses",
        "Sessions",
        "Fold count",
        "Fold IDs",
        "Missing assignments",
        "Extra assignments",
        "Duplicate response assignments",
        "Response-session mismatches",
        "Cross-fold sessions",
        "Maximum folds per session",
        "Assignment mismatches",
        "Required failures",
        "FROZEN_FOLD_CONTRACT_READY",
    ],
    "value": [
        len(responses_base),
        len(frozen_folds),
        len(session_fold_audit),
        len(observed_folds),
        ",".join(map(str, sorted(observed_folds))),
        len(missing_fold_responses),
        len(extra_fold_responses),
        duplicate_response_rows,
        response_session_mismatches,
        len(cross_fold_sessions),
        int(session_fold_audit["fold_count"].max()),
        len(foundation_only_assignments) + len(manifest_only_assignments),
        len(fold_failures),
        FROZEN_FOLD_CONTRACT_READY,
    ],
})

display(fold_checks)
display(fold_gate_summary)

assert FROZEN_FOLD_CONTRACT_READY, (
    "Section 1.9 failed.\n\n"
    + fold_failures[["check", "detail"]].to_string(index=False)
)

,check,required,passed,detail
0,Every foundation response has a fold assignment,True,True,0
1,No extra response exists in fold manifest,True,True,0
2,No duplicate response assignment,True,True,0
3,Response-session mapping is consistent,True,True,0
4,Each response maps to one session,True,True,0
5,Each response maps to one fold,True,True,0
6,No session crosses folds,True,True,0
7,Maximum one fold per session,True,True,1
8,Fold domain is exactly 0–4,True,True,"observed=[np.int64(0), np.int64(1), np.int64(2..."
9,No null fold values,True,True,0


,item,value
0,Foundation responses,35072
1,Manifest responses,35072
2,Sessions,22821
3,Fold count,5
4,Fold IDs,"0,1,2,3,4"
5,Missing assignments,0
6,Extra assignments,0
7,Duplicate response assignments,0
8,Response-session mismatches,0
9,Cross-fold sessions,0


# Section 1.10 — Create the Data Contract

This section converts the verified inventory into a machine-readable parser contract.
The contract binds the parser to the verified source fingerprints and frozen fold manifest.
Raw source fields and response-foundation fields are defined separately.
Transcript input fields, naming rules, normalization policy, and ordering policy are frozen explicitly.
Raw values must be preserved while derived fields are created separately.
Turn identity must be deterministic, machine-independent, and traceable to the original source.
The parser is explicitly label-blind and cannot use target-derived or test-population statistics.
Candidate parser outputs are defined but are not considered canonical.
The contract is validated before writing and verified again after reloading from disk.
The section ends with the `DATA_CONTRACT_READY` gate.

In [35]:
assert FROZEN_FOLD_CONTRACT_READY, "Section 1.9 must pass before Section 1.10."

TRANSCRIPT_REQUIRED_FIELDS = [
    "session_id",
    "utterance_id",
    "role",
    "content",
    "timestamp",
]

RESPONSE_REQUIRED_FIELDS = [
    "response_id",
    "session_id",
    "objective_id_raw",
    "objective_raw",
    "target",
    "fold",
]

DATA_CONTRACT_PATH = PHASE1_ROOT / "data_contract.json"

DATA_CONTRACT = {
    "contract": {
        "name": "trace_the_ace_data_foundation",
        "version": "1.0",
        "phase": "data_foundation",
        "created_by": "01_data_inventory.ipynb",
        "run_id": RUN_ID,
        "path_resolution": "path_registry.json",
    },

    "source_binding": {
        "train_features_sha256": current_source_fingerprints["train_features_sha256"],
        "train_labels_sha256": current_source_fingerprints["train_labels_sha256"],
        "frozen_fold_manifest_sha256": current_source_fingerprints["frozen_fold_manifest_sha256"],
        "transcript_directory_sha256": current_source_fingerprints["transcript_directory_sha256"],
        "expected_response_count": int(len(responses_base)),
        "expected_session_count": int(responses_base["session_id"].nunique()),
        "expected_transcript_file_count": int(len(transcript_file_inventory)),
    },

    "raw_source_schema": {
        "response_key": RESOLVED_SCHEMA["response_key"],
        "session_key": RESOLVED_SCHEMA["session_key"],
        "objective_id_field": RESOLVED_SCHEMA["objective_id_field"],
        "objective_field": RESOLVED_SCHEMA["objective_field"],
        "target_field": RESOLVED_SCHEMA["target_field"],
        "fold_field": RESOLVED_SCHEMA["fold_field"],
    },

    "response_schema": {
        "unit": "one_row_per_session_objective_response",
        "required_fields": RESPONSE_REQUIRED_FIELDS,
        "response_key": "response_id",
        "session_key": "session_id",
        "objective_id_field": "objective_id_raw",
        "objective_field": "objective_raw",
        "target_field": "target",
        "fold_field": "fold",
    },

    "transcript_schema": {
        "unit": "one_row_per_raw_utterance",
        "required_fields": TRANSCRIPT_REQUIRED_FIELDS,
        "allow_extra_columns": True,
    },

    "field_naming": {
        "session_id": "session_id_raw",
        "utterance_id": "utterance_id_raw",
        "role": "role_raw",
        "content": "content_raw",
        "timestamp": "timestamp_raw",
        "normalized_text": "text_norm",
        "parsed_utterance_id": "utterance_id",
        "canonical_role": "role",
        "parsed_timestamp": "timestamp",
    },

    "text_normalization": {
        "version": "1.0",
        "unicode_form": "NFKC",
        "normalize_line_endings": True,
        "trim_outer_whitespace": True,
        "collapse_repeated_whitespace": True,
        "forbidden_operations": [
            "stopword_removal",
            "stemming",
            "punctuation_stripping",
            "number_rewriting",
            "spell_correction",
            "llm_rewriting",
            "unclear_marker_deletion",
            "semantic_rewriting",
        ],
    },

    "role_policy": {
        "preserve_raw": True,
        "inspect_values_before_mapping": True,
        "formatting_normalization_only": True,
        "lexical_role_inference": False,
        "unknown_role_value": "unknown",
    },

    "ordering_policy": {
        "version": "1.0",
        "primary": [
            "timestamp",
            "utterance_id",
            "source_file_relative",
            "source_row_index",
        ],
        "timestamp_fallback": [
            "utterance_id",
            "source_file_relative",
            "source_row_index",
        ],
        "final_fallback": [
            "source_file_relative",
            "source_row_index",
        ],
        "silent_repair": False,
        "required_outputs": [
            "ordering_method",
            "ordering_confidence",
            "ordering_issue_flag",
        ],
    },

    "turn_identity": {
        "version": "1.0",
        "deterministic": True,
        "machine_independent": True,
        "absolute_path_allowed": False,
        "preserve_raw_row_hash": True,
        "preserve_content_hash": True,
    },

    "provenance": {
        "required_fields": [
            "source_file_relative",
            "source_row_index",
            "session_id_raw",
            "utterance_id_raw",
            "role_raw",
            "content_raw",
            "timestamp_raw",
        ]
    },

    "fold_contract": {
        "fold_count": len(EXPECTED_FOLDS),
        "fold_ids": sorted(int(x) for x in EXPECTED_FOLDS),
        "session_grouped": True,
        "cross_fold_session_allowed": False,
        "manifest_sha256": FROZEN_FOLD_SHA256,
    },

    "parser_restrictions": {
        "label_blind": True,
        "forbidden_inputs": [
            "target",
            "objective_prior",
            "oof_prediction",
            "baseline_prediction",
            "validation_metric",
            "test_population_statistics",
        ],
    },

    "candidate_outputs": {
        "candidate_is_canonical": False,
        "turns": "02_turn_parser/turns_candidate.parquet",
        "sessions": "02_turn_parser/sessions_candidate.parquet",
        "manifest": "02_turn_parser/parser_manifest.json",
    },
}

In [36]:
def valid_sha256(value):
    if not isinstance(value, str) or len(value) != 64:
        return False

    try:
        int(value, 16)
        return True
    except ValueError:
        return False


required_blocks = {
    "contract",
    "source_binding",
    "raw_source_schema",
    "response_schema",
    "transcript_schema",
    "field_naming",
    "text_normalization",
    "role_policy",
    "ordering_policy",
    "turn_identity",
    "provenance",
    "fold_contract",
    "parser_restrictions",
    "candidate_outputs",
}

source_hashes = [
    DATA_CONTRACT["source_binding"]["train_features_sha256"],
    DATA_CONTRACT["source_binding"]["train_labels_sha256"],
    DATA_CONTRACT["source_binding"]["frozen_fold_manifest_sha256"],
    DATA_CONTRACT["source_binding"]["transcript_directory_sha256"],
]

dominant_transcript_columns = set(
    transcript_file_inventory.loc[
        transcript_file_inventory["schema_status"].isin(
            ["VALID", "VALID_WITH_EXTRA_COLUMNS"]
        ),
        "columns_text",
    ].iloc[0].split(" | ")
)

contract_text = json.dumps(DATA_CONTRACT, ensure_ascii=False)

contract_checks = pd.DataFrame([
    check_row(
        "All required contract blocks exist",
        required_blocks.issubset(DATA_CONTRACT),
        sorted(required_blocks - set(DATA_CONTRACT))
    ),
    check_row(
        "All source fingerprints are valid SHA256",
        all(valid_sha256(value) for value in source_hashes),
        sum(not valid_sha256(value) for value in source_hashes)
    ),
    check_row(
        "Response contract matches responses_base",
        set(RESPONSE_REQUIRED_FIELDS).issubset(responses_base.columns),
        sorted(set(RESPONSE_REQUIRED_FIELDS) - set(responses_base.columns))
    ),
    check_row(
        "Transcript required fields match verified schema",
        set(TRANSCRIPT_REQUIRED_FIELDS).issubset(dominant_transcript_columns),
        sorted(set(TRANSCRIPT_REQUIRED_FIELDS) - dominant_transcript_columns)
    ),
    check_row(
        "Expected response count matches",
        DATA_CONTRACT["source_binding"]["expected_response_count"] == len(responses_base),
        DATA_CONTRACT["source_binding"]["expected_response_count"]
    ),
    check_row(
        "Expected session count matches",
        DATA_CONTRACT["source_binding"]["expected_session_count"] == responses_base["session_id"].nunique(),
        DATA_CONTRACT["source_binding"]["expected_session_count"]
    ),
    check_row(
        "Expected transcript file count matches",
        DATA_CONTRACT["source_binding"]["expected_transcript_file_count"] == len(transcript_file_inventory),
        DATA_CONTRACT["source_binding"]["expected_transcript_file_count"]
    ),
    check_row(
        "Frozen fold IDs match verified domain",
        DATA_CONTRACT["fold_contract"]["fold_ids"] == sorted(EXPECTED_FOLDS),
        DATA_CONTRACT["fold_contract"]["fold_ids"]
    ),
    check_row(
        "Session grouping is mandatory",
        DATA_CONTRACT["fold_contract"]["session_grouped"]
        and not DATA_CONTRACT["fold_contract"]["cross_fold_session_allowed"],
        DATA_CONTRACT["fold_contract"]
    ),
    check_row(
        "Parser is label-blind",
        DATA_CONTRACT["parser_restrictions"]["label_blind"],
        DATA_CONTRACT["parser_restrictions"]["label_blind"]
    ),
    check_row(
        "Silent ordering repair is prohibited",
        not DATA_CONTRACT["ordering_policy"]["silent_repair"],
        DATA_CONTRACT["ordering_policy"]["silent_repair"]
    ),
    check_row(
        "Turn identity is machine-independent",
        DATA_CONTRACT["turn_identity"]["machine_independent"]
        and not DATA_CONTRACT["turn_identity"]["absolute_path_allowed"],
        DATA_CONTRACT["turn_identity"]
    ),
    check_row(
        "Candidate outputs are not canonical",
        not DATA_CONTRACT["candidate_outputs"]["candidate_is_canonical"],
        DATA_CONTRACT["candidate_outputs"]["candidate_is_canonical"]
    ),
    check_row(
        "Project absolute path is not embedded",
        str(PROJECT_ROOT) not in contract_text
        and str(TRANSCRIPT_ROOT) not in contract_text,
        "absolute_path_found" if str(PROJECT_ROOT) in contract_text or str(TRANSCRIPT_ROOT) in contract_text else "clean"
    ),
])

contract_failures = contract_checks[
    contract_checks["required"] & ~contract_checks["passed"]
]

DATA_CONTRACT_VALID = contract_failures.empty

display(contract_checks)

print(f"DATA_CONTRACT_VALID: {DATA_CONTRACT_VALID}")

assert DATA_CONTRACT_VALID, (
    "Section 1.10 contract validation failed.\n\n"
    + contract_failures[["check", "detail"]].to_string(index=False)
)

,check,required,passed,detail
0,All required contract blocks exist,True,True,[]
1,All source fingerprints are valid SHA256,True,True,0
2,Response contract matches responses_base,True,True,[]
3,Transcript required fields match verified schema,True,True,[]
4,Expected response count matches,True,True,35072
5,Expected session count matches,True,True,22821
6,Expected transcript file count matches,True,True,22821
7,Frozen fold IDs match verified domain,True,True,"[0, 1, 2, 3, 4]"
8,Session grouping is mandatory,True,True,"{'fold_count': 5, 'fold_ids': [0, 1, 2, 3, 4],..."
9,Parser is label-blind,True,True,True


DATA_CONTRACT_VALID: True


In [37]:
def canonical_json_hash(data):
    text = json.dumps(
        data,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


with open(DATA_CONTRACT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        DATA_CONTRACT,
        f,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )

with open(DATA_CONTRACT_PATH, "r", encoding="utf-8") as f:
    reloaded_contract = json.load(f)

contract_reload_identical = DATA_CONTRACT == reloaded_contract

DATA_CONTRACT_SHA256 = canonical_json_hash(DATA_CONTRACT)
reloaded_contract_sha256 = canonical_json_hash(reloaded_contract)

contract_hash_identical = (
    DATA_CONTRACT_SHA256 == reloaded_contract_sha256
)

DATA_CONTRACT_READY = (
    DATA_CONTRACT_VALID
    and DATA_CONTRACT_PATH.exists()
    and contract_reload_identical
    and contract_hash_identical
)

contract_summary = pd.DataFrame({
    "item": [
        "Contract version",
        "Response key",
        "Session key",
        "Objective field",
        "Target field",
        "Transcript required fields",
        "Expected responses",
        "Expected sessions",
        "Expected transcript files",
        "Fold count",
        "Fold IDs",
        "Parser label-blind",
        "Candidate is canonical",
        "Contract validation failures",
        "Reload identical",
        "Hash identical",
        "DATA_CONTRACT_SHA256",
        "DATA_CONTRACT_READY",
    ],
    "value": [
        DATA_CONTRACT["contract"]["version"],
        DATA_CONTRACT["response_schema"]["response_key"],
        DATA_CONTRACT["response_schema"]["session_key"],
        DATA_CONTRACT["response_schema"]["objective_field"],
        DATA_CONTRACT["response_schema"]["target_field"],
        len(DATA_CONTRACT["transcript_schema"]["required_fields"]),
        DATA_CONTRACT["source_binding"]["expected_response_count"],
        DATA_CONTRACT["source_binding"]["expected_session_count"],
        DATA_CONTRACT["source_binding"]["expected_transcript_file_count"],
        DATA_CONTRACT["fold_contract"]["fold_count"],
        ",".join(map(str, DATA_CONTRACT["fold_contract"]["fold_ids"])),
        DATA_CONTRACT["parser_restrictions"]["label_blind"],
        DATA_CONTRACT["candidate_outputs"]["candidate_is_canonical"],
        len(contract_failures),
        contract_reload_identical,
        contract_hash_identical,
        DATA_CONTRACT_SHA256,
        DATA_CONTRACT_READY,
    ],
})

display(contract_summary)

assert DATA_CONTRACT_READY, (
    "Section 1.10 failed: data_contract.json did not survive validation and reload."
)

print(f"Saved: {DATA_CONTRACT_PATH}")

,item,value
0,Contract version,1.0
1,Response key,response_id
2,Session key,session_id
3,Objective field,objective_raw
4,Target field,target
5,Transcript required fields,5
6,Expected responses,35072
7,Expected sessions,22821
8,Expected transcript files,22821
9,Fold count,5


Saved: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\01_data_foundation\data_contract.json


# Section 1.11 — Inventory Audit

This section performs the final audit of all inventory work completed in Sections 1.0–1.10.
No new dataset transformation or source discovery is performed here.
The audit is organized around correctness, coverage, failure analysis, and downstream readiness.
All previously validated stage gates are rechecked together.
Known non-blocking issues are documented separately from blocking failures.
The saved response foundation and data contract are verified as parser inputs.
Transcript and frozen-fold coverage must remain complete.
Historical objective reconciliation is recorded without redefining the current objective population.
Passing this section means the inventory is ready for the final hard gate.
The section ends with `INVENTORY_AUDIT_READY`, not the final `INVENTORY_READY`.

In [38]:
assert DATA_CONTRACT_READY, "Section 1.10 must pass before Section 1.11."

stage_gates = pd.DataFrame({
    "section": [
        "1.0",
        "1.1",
        "1.2",
        "1.3",
        "1.4",
        "1.5",
        "1.6",
        "1.7",
        "1.8",
        "1.9",
        "1.10",
    ],
    "gate": [
        "FOUNDATION_BOOTSTRAP_READY",
        "SOURCE_AUTHORITY_READY",
        "SOURCE_IDENTITY_READY",
        "SCHEMA_KEYS_VALID",
        "RESPONSE_FOUNDATION_READY",
        "POPULATION_CENSUS_READY",
        "OBJECTIVE_RECONCILIATION_READY",
        "TRANSCRIPT_INVENTORY_READY",
        "TRANSCRIPT_COVERAGE_READY",
        "FROZEN_FOLD_CONTRACT_READY",
        "DATA_CONTRACT_READY",
    ],
    "passed": [
        FOUNDATION_BOOTSTRAP_READY,
        SOURCE_AUTHORITY_READY,
        SOURCE_IDENTITY_READY,
        SCHEMA_KEYS_VALID,
        RESPONSE_FOUNDATION_READY,
        POPULATION_CENSUS_READY,
        OBJECTIVE_RECONCILIATION_READY,
        TRANSCRIPT_INVENTORY_READY,
        TRANSCRIPT_COVERAGE_READY,
        FROZEN_FOLD_CONTRACT_READY,
        DATA_CONTRACT_READY,
    ],
})

stage_gates["status"] = stage_gates["passed"].map({
    True: "PASS",
    False: "FAIL",
})

failed_stage_gates = stage_gates[~stage_gates["passed"]]

display(stage_gates)

print(
    f"Passed stages: {stage_gates['passed'].sum()} / {len(stage_gates)}"
)

,section,gate,passed,status
0,1.0,FOUNDATION_BOOTSTRAP_READY,True,PASS
1,1.1,SOURCE_AUTHORITY_READY,True,PASS
2,1.2,SOURCE_IDENTITY_READY,True,PASS
3,1.3,SCHEMA_KEYS_VALID,True,PASS
4,1.4,RESPONSE_FOUNDATION_READY,True,PASS
5,1.5,POPULATION_CENSUS_READY,True,PASS
6,1.6,OBJECTIVE_RECONCILIATION_READY,True,PASS
7,1.7,TRANSCRIPT_INVENTORY_READY,True,PASS
8,1.8,TRANSCRIPT_COVERAGE_READY,True,PASS
9,1.9,FROZEN_FOLD_CONTRACT_READY,True,PASS


Passed stages: 11 / 11


In [39]:
saved_responses_ready = RESPONSES_BASE_PATH.exists()
saved_contract_ready = DATA_CONTRACT_PATH.exists()

saved_responses_rows = (
    len(pd.read_parquet(RESPONSES_BASE_PATH, columns=["response_id"]))
    if saved_responses_ready else None
)

objective_whitespace_rows = text_issue_counts(
    responses_base["objective_raw"]
)[1]

historical_reconciliation_ok = (
    HISTORICAL_OBJECTIVE_RECONCILIATION
    in {
        "EXPLAINED_AS_HISTORICAL_COLLAPSE",
        "REFERENCE_UNAVAILABLE",
        "UNRESOLVED",
    }
)

inventory_audit_checks = pd.DataFrame([
    # Correctness
    check_row(
        "Correctness — all stage gates passed",
        failed_stage_gates.empty,
        len(failed_stage_gates)
    ),
    check_row(
        "Correctness — response foundation remains unique",
        responses_base["response_id"].is_unique,
        int(responses_base["response_id"].duplicated().sum())
    ),
    check_row(
        "Correctness — current objective population valid",
        CURRENT_OBJECTIVE_POPULATION_VALID,
        current_id_count
    ),
    check_row(
        "Correctness — data contract valid",
        DATA_CONTRACT_READY,
        DATA_CONTRACT_SHA256
    ),

    # Coverage
    check_row(
        "Coverage — all labelled sessions have usable transcripts",
        missing_sessions == 0
        and unusable_sessions == 0
        and ambiguous_sessions == 0,
        (
            f"missing={missing_sessions}, "
            f"unusable={unusable_sessions}, "
            f"ambiguous={ambiguous_sessions}"
        )
    ),
    check_row(
        "Coverage — all response rows have transcript coverage",
        matched_response_rows == len(responses_base)
        and uncovered_response_rows == 0,
        f"{matched_response_rows} / {len(responses_base)}"
    ),
    check_row(
        "Coverage — frozen fold contract complete",
        FROZEN_FOLD_CONTRACT_READY,
        f"cross_fold_sessions={len(cross_fold_sessions)}"
    ),

    # Failure analysis
    check_row(
        "Failure analysis — historical objective difference documented",
        historical_reconciliation_ok,
        HISTORICAL_OBJECTIVE_RECONCILIATION,
        required=False
    ),
    check_row(
        "Failure analysis — objective boundary whitespace absent",
        objective_whitespace_rows == 0,
        objective_whitespace_rows,
        required=False
    ),
    check_row(
        "Failure analysis — no orphan transcript sessions",
        orphan_sessions == 0,
        orphan_sessions,
        required=False
    ),

    # Downstream readiness
    check_row(
        "Downstream — responses_base artifact exists",
        saved_responses_ready,
        str(RESPONSES_BASE_PATH)
    ),
    check_row(
        "Downstream — saved response row count matches memory",
        saved_responses_rows == len(responses_base),
        f"disk={saved_responses_rows}, memory={len(responses_base)}"
    ),
    check_row(
        "Downstream — data_contract.json exists",
        saved_contract_ready,
        str(DATA_CONTRACT_PATH)
    ),
    check_row(
        "Downstream — parser contract is label-blind",
        DATA_CONTRACT["parser_restrictions"]["label_blind"],
        DATA_CONTRACT["parser_restrictions"]["label_blind"]
    ),
    check_row(
        "Downstream — candidate outputs remain non-canonical",
        not DATA_CONTRACT["candidate_outputs"]["candidate_is_canonical"],
        DATA_CONTRACT["candidate_outputs"]["candidate_is_canonical"]
    ),
])

audit_failures = inventory_audit_checks[
    inventory_audit_checks["required"]
    & ~inventory_audit_checks["passed"]
]

audit_warnings = inventory_audit_checks[
    ~inventory_audit_checks["required"]
    & ~inventory_audit_checks["passed"]
]

display(inventory_audit_checks)

,check,required,passed,detail
0,Correctness — all stage gates passed,True,True,0
1,Correctness — response foundation remains unique,True,True,0
2,Correctness — current objective population valid,True,True,398
3,Correctness — data contract valid,True,True,23c1f8fe4bcb16439bb1a74928b603e6b2e8aa59774fc4...
4,Coverage — all labelled sessions have usable t...,True,True,"missing=0, unusable=0, ambiguous=0"
5,Coverage — all response rows have transcript c...,True,True,35072 / 35072
6,Coverage — frozen fold contract complete,True,True,cross_fold_sessions=0
7,Failure analysis — historical objective differ...,False,True,EXPLAINED_AS_HISTORICAL_COLLAPSE
8,Failure analysis — objective boundary whitespa...,False,False,3809
9,Failure analysis — no orphan transcript sessions,False,True,0


In [40]:
audit_domains = {
    "CORRECTNESS": [
        "Correctness — all stage gates passed",
        "Correctness — response foundation remains unique",
        "Correctness — current objective population valid",
        "Correctness — data contract valid",
    ],
    "COVERAGE": [
        "Coverage — all labelled sessions have usable transcripts",
        "Coverage — all response rows have transcript coverage",
        "Coverage — frozen fold contract complete",
    ],
    "FAILURE_ANALYSIS": [
        "Failure analysis — historical objective difference documented",
        "Failure analysis — objective boundary whitespace absent",
        "Failure analysis — no orphan transcript sessions",
    ],
    "DOWNSTREAM_READINESS": [
        "Downstream — responses_base artifact exists",
        "Downstream — saved response row count matches memory",
        "Downstream — data_contract.json exists",
        "Downstream — parser contract is label-blind",
        "Downstream — candidate outputs remain non-canonical",
    ],
}

domain_rows = []

for domain, checks in audit_domains.items():
    subset = inventory_audit_checks[
        inventory_audit_checks["check"].isin(checks)
    ]

    required_subset = subset[subset["required"]]
    required_pass = bool(required_subset["passed"].all())

    domain_rows.append({
        "domain": domain,
        "checks": len(subset),
        "required_failures": int(
            (required_subset["passed"] == False).sum()
        ),
        "warnings": int(
            ((subset["required"] == False) & (subset["passed"] == False)).sum()
        ),
        "status": "PASS" if required_pass else "FAIL",
    })

inventory_audit_domains = pd.DataFrame(domain_rows)

INVENTORY_AUDIT_READY = (
    failed_stage_gates.empty
    and audit_failures.empty
)

inventory_audit_summary = pd.DataFrame({
    "item": [
        "Completed stage gates",
        "Passed stage gates",
        "Authoritative responses",
        "Authoritative sessions",
        "Current objectives",
        "Matched transcript sessions",
        "Matched response rows",
        "Cross-fold sessions",
        "Historical reconciliation",
        "Required audit failures",
        "Non-blocking audit warnings",
        "Data contract ready",
        "INVENTORY_AUDIT_READY",
    ],
    "value": [
        len(stage_gates),
        int(stage_gates["passed"].sum()),
        len(responses_base),
        responses_base["session_id"].nunique(),
        current_id_count,
        int(session_coverage["coverage_status"].eq("MATCHED").sum()),
        matched_response_rows,
        len(cross_fold_sessions),
        HISTORICAL_OBJECTIVE_RECONCILIATION,
        len(audit_failures),
        len(audit_warnings),
        DATA_CONTRACT_READY,
        INVENTORY_AUDIT_READY,
    ],
})

display(inventory_audit_domains)
display(inventory_audit_summary)

assert INVENTORY_AUDIT_READY, (
    "Section 1.11 failed.\n\n"
    + audit_failures[["check", "detail"]].to_string(index=False)
)

,domain,checks,required_failures,warnings,status
0,CORRECTNESS,4,0,0,PASS
1,COVERAGE,3,0,0,PASS
2,FAILURE_ANALYSIS,3,0,1,PASS
3,DOWNSTREAM_READINESS,5,0,0,PASS


,item,value
0,Completed stage gates,11
1,Passed stage gates,11
2,Authoritative responses,35072
3,Authoritative sessions,22821
4,Current objectives,398
5,Matched transcript sessions,22821
6,Matched response rows,35072
7,Cross-fold sessions,0
8,Historical reconciliation,EXPLAINED_AS_HISTORICAL_COLLAPSE
9,Required audit failures,0


# Section 1.12 — Final Inventory Gate & Freeze

This section performs the final hard gate for the Data Inventory notebook.
All critical source, response, objective, transcript, fold, and contract conditions are verified again.
Known non-blocking data issues are recorded separately as warnings.
The saved response foundation is reloaded and compared with the verified in-memory table.
A compact `inventory_manifest.json` records population, fingerprints, artifacts, gates, and warnings.
Only project-relative artifact paths are stored in the manifest.
The manifest is saved, reloaded, and hashed before the inventory is considered frozen.
Warnings do not block progression unless they violate a required foundation contract.
No new data transformation or normalization is performed here.
The notebook ends only when `INVENTORY_READY = True`.

In [41]:
assert INVENTORY_AUDIT_READY, "Section 1.11 must pass before Section 1.12."

# Recheck saved response artifact
saved_responses = pd.read_parquet(RESPONSES_BASE_PATH) if RESPONSES_BASE_PATH.exists() else None

response_artifact_identical = (
    saved_responses is not None
    and list(saved_responses.columns) == list(responses_base.columns)
    and saved_responses.reset_index(drop=True).equals(responses_base.reset_index(drop=True))
)

target_domain = set(responses_base["target"].dropna().unique())
objective_whitespace_rows = text_issue_counts(responses_base["objective_raw"])[1]

final_inventory_checks = pd.DataFrame([
    check_row("All previous stage gates passed", stage_gates["passed"].all(), int((~stage_gates["passed"]).sum())),
    check_row("Inventory audit passed", INVENTORY_AUDIT_READY, INVENTORY_AUDIT_READY),
    check_row("Response foundation is unique", responses_base["response_id"].is_unique, int(responses_base["response_id"].duplicated().sum())),
    check_row("Target domain is valid", target_domain.issubset({0, 1}), sorted(target_domain)),
    check_row("Current objective population is valid", CURRENT_OBJECTIVE_POPULATION_VALID, current_id_count),
    check_row("All labelled sessions have usable transcripts", missing_sessions == 0 and unusable_sessions == 0 and ambiguous_sessions == 0, f"missing={missing_sessions}, unusable={unusable_sessions}, ambiguous={ambiguous_sessions}"),
    check_row("All response rows have transcript coverage", matched_response_rows == len(responses_base) and uncovered_response_rows == 0, f"{matched_response_rows} / {len(responses_base)}"),
    check_row("Frozen fold contract passed", FROZEN_FOLD_CONTRACT_READY, f"cross_fold_sessions={len(cross_fold_sessions)}"),
    check_row("Data contract passed", DATA_CONTRACT_READY, DATA_CONTRACT_SHA256),
    check_row("responses_base.parquet exists", RESPONSES_BASE_PATH.exists(), str(RESPONSES_BASE_PATH)),
    check_row("Saved response foundation matches memory", response_artifact_identical, response_artifact_identical),
    check_row("data_contract.json exists", DATA_CONTRACT_PATH.exists(), str(DATA_CONTRACT_PATH)),
])

inventory_blockers = final_inventory_checks[
    final_inventory_checks["required"] & ~final_inventory_checks["passed"]
]

warning_records = []

if objective_whitespace_rows > 0:
    warning_records.append({
        "code": "OBJECTIVE_BOUNDARY_WHITESPACE",
        "severity": "WARNING",
        "count": int(objective_whitespace_rows),
        "action": "PRESERVE_RAW",
        "downstream": "Use safe normalization only where explicitly required",
    })

if HISTORICAL_OBJECTIVE_RECONCILIATION != "EXPLAINED_AS_HISTORICAL_COLLAPSE":
    warning_records.append({
        "code": "HISTORICAL_OBJECTIVE_RECONCILIATION",
        "severity": "WARNING",
        "count": 1,
        "action": "DOCUMENT_ONLY",
        "downstream": HISTORICAL_OBJECTIVE_RECONCILIATION,
    })

if orphan_sessions > 0:
    warning_records.append({
        "code": "ORPHAN_TRANSCRIPT_SESSIONS",
        "severity": "WARNING",
        "count": int(orphan_sessions),
        "action": "DO_NOT_USE_FOR_TRAINING",
        "downstream": "Retain for provenance only",
    })

inventory_warnings = pd.DataFrame(warning_records)
FINAL_GATE_PASS = inventory_blockers.empty

display(final_inventory_checks)
display(inventory_warnings)

print(f"Final blockers : {len(inventory_blockers)}")
print(f"Final warnings : {len(inventory_warnings)}")
print(f"FINAL_GATE_PASS: {FINAL_GATE_PASS}")

assert FINAL_GATE_PASS, (
    "Section 1.12 hard gate failed.\n\n"
    + inventory_blockers[["check", "detail"]].to_string(index=False)
)

,check,required,passed,detail
0,All previous stage gates passed,True,True,0
1,Inventory audit passed,True,True,True
2,Response foundation is unique,True,True,0
3,Target domain is valid,True,True,"[0.0, 1.0]"
4,Current objective population is valid,True,True,398
5,All labelled sessions have usable transcripts,True,True,"missing=0, unusable=0, ambiguous=0"
6,All response rows have transcript coverage,True,True,35072 / 35072
7,Frozen fold contract passed,True,True,cross_fold_sessions=0
8,Data contract passed,True,True,23c1f8fe4bcb16439bb1a74928b603e6b2e8aa59774fc4...
9,responses_base.parquet exists,True,True,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...


,code,severity,count,action,downstream
0,OBJECTIVE_BOUNDARY_WHITESPACE,WARNING,3809,PRESERVE_RAW,Use safe normalization only where explicitly r...


Final blockers : 0
Final warnings : 1
FINAL_GATE_PASS: True


In [42]:
def relative_to_phase(path):
    return Path(path).resolve().relative_to(PHASE1_ROOT.resolve()).as_posix()

def relative_to_project(path):
    path = Path(path).resolve()

    try:
        return path.relative_to(PROJECT_ROOT.resolve()).as_posix()
    except ValueError:
        return None

RESPONSES_BASE_SHA256 = sha256_file(RESPONSES_BASE_PATH)
INVENTORY_MANIFEST_PATH = INVENTORY_OUTPUT_DIR / "inventory_manifest.json"

manifest_stage_gates = json.loads(stage_gates.to_json(orient="records"))
manifest_stage_gates.append({
    "section": "1.11",
    "gate": "INVENTORY_AUDIT_READY",
    "passed": bool(INVENTORY_AUDIT_READY),
    "status": "PASS" if INVENTORY_AUDIT_READY else "FAIL",
})

INVENTORY_MANIFEST = {
    "inventory": {
        "name": "trace_the_ace_data_inventory",
        "version": "1.0",
        "run_id": RUN_ID,
        "status": "PASS",
        "inventory_ready": bool(FINAL_GATE_PASS),
        "contract_version": DATA_CONTRACT["contract"]["version"],
    },

    "population": {
        "responses": int(len(responses_base)),
        "sessions": int(responses_base["session_id"].nunique()),
        "objective_ids": int(current_id_count),
        "raw_objective_texts": int(current_raw_count),
        "safe_objective_texts": int(current_safe_count),
        "case_objective_keys": int(current_case_count),
        "positive_rows": int(n_positive),
        "negative_rows": int(n_negative),
    },

    "objective_reconciliation": {
        "current_authoritative_count": int(current_id_count),
        "historical_count": int(historical_objective_count) if historical_objective_count is not None else None,
        "status": HISTORICAL_OBJECTIVE_RECONCILIATION,
    },

    "transcripts": {
        "source_type": path_registry["transcript_source_type"],
        "source_relative": relative_to_project(TRANSCRIPT_ROOT),
        "files": int(len(transcript_file_inventory)),
        "internal_sessions": int(len(transcript_session_population)),
        "matched_sessions": int(session_coverage["coverage_status"].eq("MATCHED").sum()),
        "missing_sessions": int(missing_sessions),
        "unusable_sessions": int(unusable_sessions),
        "ambiguous_multi_source_sessions": int(ambiguous_sessions),
        "orphan_sessions": int(orphan_sessions),
    },

    "folds": {
        "fold_count": int(len(observed_folds)),
        "fold_ids": sorted(int(x) for x in observed_folds),
        "cross_fold_sessions": int(len(cross_fold_sessions)),
        "max_folds_per_session": int(session_fold_audit["fold_count"].max()),
        "manifest_sha256": FROZEN_FOLD_SHA256,
    },

    "artifacts": {
        "responses_base": relative_to_phase(RESPONSES_BASE_PATH),
        "data_contract": relative_to_phase(DATA_CONTRACT_PATH),
        "responses_base_sha256": RESPONSES_BASE_SHA256,
        "data_contract_sha256": DATA_CONTRACT_SHA256,
    },

    "source_fingerprints": {
        "train_features_sha256": current_source_fingerprints["train_features_sha256"],
        "train_labels_sha256": current_source_fingerprints["train_labels_sha256"],
        "frozen_fold_manifest_sha256": current_source_fingerprints["frozen_fold_manifest_sha256"],
        "transcript_directory_sha256": current_source_fingerprints["transcript_directory_sha256"],
    },

    "stage_gates": manifest_stage_gates,
    "blocking_failures": int(len(inventory_blockers)),
    "warnings": warning_records,
}

with open(INVENTORY_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(INVENTORY_MANIFEST, f, ensure_ascii=False, indent=2, sort_keys=True)

print(f"Saved: {INVENTORY_MANIFEST_PATH}")

Saved: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\01_data_foundation\01_inventory\inventory_manifest.json


In [43]:
with open(INVENTORY_MANIFEST_PATH, "r", encoding="utf-8") as f:
    reloaded_inventory_manifest = json.load(f)

manifest_reload_identical = INVENTORY_MANIFEST == reloaded_inventory_manifest

INVENTORY_MANIFEST_SHA256 = canonical_json_hash(INVENTORY_MANIFEST)
reloaded_manifest_sha256 = canonical_json_hash(reloaded_inventory_manifest)

manifest_hash_identical = INVENTORY_MANIFEST_SHA256 == reloaded_manifest_sha256

required_inventory_artifacts = {
    "responses_base.parquet": RESPONSES_BASE_PATH,
    "data_contract.json": DATA_CONTRACT_PATH,
    "inventory_manifest.json": INVENTORY_MANIFEST_PATH,
}

artifact_status = pd.DataFrame([
    {
        "artifact": name,
        "exists": path.exists(),
        "relative_path": relative_to_phase(path),
    }
    for name, path in required_inventory_artifacts.items()
])

all_artifacts_exist = artifact_status["exists"].all()

INVENTORY_READY = (
    FINAL_GATE_PASS
    and INVENTORY_AUDIT_READY
    and DATA_CONTRACT_READY
    and response_artifact_identical
    and all_artifacts_exist
    and manifest_reload_identical
    and manifest_hash_identical
)

final_inventory_summary = pd.DataFrame({
    "item": [
        "Responses",
        "Sessions",
        "Current objectives",
        "Transcript sessions matched",
        "Response rows matched",
        "Cross-fold sessions",
        "Final blockers",
        "Final warnings",
        "Data contract ready",
        "Response artifact verified",
        "Manifest reload identical",
        "Manifest hash identical",
        "Inventory manifest SHA256",
        "INVENTORY_READY",
    ],
    "value": [
        len(responses_base),
        responses_base["session_id"].nunique(),
        current_id_count,
        int(session_coverage["coverage_status"].eq("MATCHED").sum()),
        matched_response_rows,
        len(cross_fold_sessions),
        len(inventory_blockers),
        len(inventory_warnings),
        DATA_CONTRACT_READY,
        response_artifact_identical,
        manifest_reload_identical,
        manifest_hash_identical,
        INVENTORY_MANIFEST_SHA256,
        INVENTORY_READY,
    ],
})

display(artifact_status)
display(final_inventory_summary)

assert INVENTORY_READY, (
    "Data Inventory was not frozen successfully. "
    "Do not start 02_turn_parser.ipynb."
)

print("\n" + "=" * 60)
print("TRACE THE ACE — DATA INVENTORY COMPLETE")
print("=" * 60)
print(f"Responses        : {len(responses_base):,}")
print(f"Sessions         : {responses_base['session_id'].nunique():,}")
print(f"Objectives       : {current_id_count:,}")
print(f"Transcript match : {int(session_coverage['coverage_status'].eq('MATCHED').sum()):,}")
print(f"Cross-fold       : {len(cross_fold_sessions)}")
print(f"Blockers         : {len(inventory_blockers)}")
print(f"Warnings         : {len(inventory_warnings)}")
print(f"INVENTORY_READY  : {INVENTORY_READY}")
print("=" * 60)

,artifact,exists,relative_path
0,responses_base.parquet,True,01_inventory/responses_base.parquet
1,data_contract.json,True,data_contract.json
2,inventory_manifest.json,True,01_inventory/inventory_manifest.json


,item,value
0,Responses,35072
1,Sessions,22821
2,Current objectives,398
3,Transcript sessions matched,22821
4,Response rows matched,35072
5,Cross-fold sessions,0
6,Final blockers,0
7,Final warnings,1
8,Data contract ready,True
9,Response artifact verified,True



TRACE THE ACE — DATA INVENTORY COMPLETE
Responses        : 35,072
Sessions         : 22,821
Objectives       : 398
Transcript match : 22,821
Cross-fold       : 0
Blockers         : 0
Warnings         : 1
INVENTORY_READY  : True


# Notebook 01 Complete — Data Inventory Frozen

The authoritative response population, objective identities, transcript sources, and frozen folds have been verified.
All 35,072 responses map to 22,821 valid sessions and usable transcript sources.
The current authoritative objective population is 398, with the historical 396 count reconciled as a prior identity collapse.
The frozen five-fold manifest has zero session overlap and remains unchanged.
`responses_base.parquet`, `data_contract.json`, and `inventory_manifest.json` are now frozen inputs.
Known raw-data warnings are preserved without modifying authoritative source values.
No retrieval, semantic modelling, objective prior, or turn reconstruction has been introduced.
The inventory may be handed to the True Turn Parser only when `INVENTORY_READY = True`.

**Next notebook:** `02_turn_parser.ipynb`